# M3 newcomer activity data: check notebook

## Plain-language guide

**Why the study needs this module.** The study asks whether a newcomer who gets a reply from their mentor keeps
editing Wikipedia more than a similar newcomer who gets no reply. To answer that, we need two kinds of facts about
every newcomer who asked a question: what they did *before* the question (to find "similar" newcomers: how many
edits, on which kinds of pages, how many were undone, how old the account is) and what they did *after* it (did
they keep editing). M3 collects exactly this activity history. Without it there are no covariates and no outcomes.

**What raw data it collects, and from where.** Everything comes from the public English Wikipedia API
(`en.wikipedia.org/w/api.php`), for the 37,095 newcomers ("askers") who posted a question to a mentor (found in M2):

* the **s6 files**, an earlier bulk download made on 2026-04-23 to 24, stored by user name: user profiles
  (`list=users`), all edits (`list=usercontribs`, 3.4 GB), log entries (`list=logevents`) and abuse filter hits
  (`list=abuselog`, 5.1 GB);
* a **top-up** fetch made on 2026-10-24, by user ID, for the 1,363 askers that s6 does not have and for 2,090
  askers whose s6 history stopped too early;
* the **account creation log** entry of each person (a `newusers` log entry);
* for edits that Wikipedia marks as undone, a lookup of **when** they were undone (page histories via
  `prop=revisions`, fetched 2026-10-24 to 26).

Only records from before the data cutoff, **2026-04-20T14:00:00Z**, are kept.

**What it does with the data, step by step.**

1. Re-indexes the s6 files by user ID (names can change, IDs never do) and keeps only askers.
2. Finds askers that s6 is missing or whose s6 edits stop too early, and fetches them again by user ID.
3. Merges the two sources, removes duplicate records, and drops everything on or after the cutoff.
4. Finds, for each person, the log entry that records how the account was created (registered themselves, made by
   someone else, or came from another Wikimedia wiki).
5. For edits tagged `mw-reverted`, finds the revision that undid them and its time.

**What it produces (rows in this run).**

| file | one row is | key columns in words | rows |
|---|---|---|---|
| `mentee_profile.parquet` | one asker | user ID, where the data came from, registration time, edit counter, blocked, user groups, how the account was created, and how many edits, logs and abuse filter hits we hold | 37,095 |
| `mentee_edits/` | one edit by an asker | revision ID, page, namespace (kind of page), time, bytes changed, tags | 3,522,460 |
| `mentee_logs.parquet` | one log entry about an asker | log type and action (for example account created, blocked), time, who did it | 547,492 |
| `mentee_abuse.parquet` | one abuse filter hit (an automatic rule flagged an action) | filter, what the user tried, what the filter did, time | 96,902 |
| `data/raw/m3/revert_times*.jsonl` | one edit tagged `mw-reverted` | how the revert was found, the revert revision, its time, hours from edit to revert | 38,214 |

**Words used below.**

* A **revert** is an edit that undoes someone else's edit. Wikipedia adds the tag **`mw-reverted`** to the edit that
  was undone. The tag does not say *when* the revert happened. That matters: a covariate measured before the
  question (for example "how many of your edits were undone") must not count a revert that happened *after* the
  question, otherwise the mentor's own reply period leaks into the "before" picture. So M3 looks up the revert time.
* The **sha1 method**: every page version has a fingerprint (sha1). If a later version has the same fingerprint as a
  version from before the edit, the page was put back, so that later revision is the revert. This is the same rule
  Wikipedia itself uses.
* **Namespace** (ns): the kind of page. 0 = article, 2 = user page, 3 = user talk page, 4 = project page.
* **Cutoff**: the fixed end of the data, 2026-04-20T14:00:00Z. Nothing on or after it may appear.

**How to read this notebook.**

* All outputs below are saved from the run on the final data (cutoff 2026-04-20T14:00Z). You do not need to run
  anything; just read.
* An **invariant** is a rule that must hold for every row (for example "no edit after the cutoff"). Each one prints
  `[OK ]` if it holds and `[FAIL]` if it does not. In this run all 103 hold.
* A **tracer** is a fixed person we always look at, so the same examples can be compared across runs.
* A **random sample** is a set of people drawn at random (fixed seed, so it is repeatable). For them the notebook
  puts the raw downloaded record next to the processed value so you can see they match. The links open the same
  record on Wikipedia.
* Under each check, **"What this checks"** says what the code tests and why it matters, and **"What the output
  shows"** explains the result.

## Technical summary

**What the module does.** For every person who asked a mentor a question (an "asker", from M2), M3 builds:

* a **profile** (`mentee_profile.parquet`): registration time, edit count, block status, user groups, current name,
  the account creation log entry (how the account was made: `create` means the person registered themselves), and
  counts of edits, logs and abuse filter hits;
* all **edits** before the data cutoff (`mentee_edits/`), all **log entries** (`mentee_logs.parquet`) and all
  **abuse filter hits** (`mentee_abuse.parquet`);
* the **revert time** of edits tagged `mw-reverted` (`data/raw/m3/revert_times*.jsonl`; the lookup has finished and covers every edit that needs it).

**How it is built (the steps this notebook covers).**

1. `index_s6.py` re-indexes the s6 files (fetched 2026-04-23 to 24, stored by user name) by user ID and keeps only askers.
2. `collect_topup.py` fetches by user ID, after the cutoff, the askers that s6 does not have (`topup`), and the askers
   whose s6 edit history stops too early (`stale.json`; these become `s6+topup`). It also looks up account creation
   logs by registration time (`creation.jsonl`).
3. `merge_activity.py` merges s6 and the top-up, removes duplicates, drops anything on or after the cutoff
   (2026-04-20T14:00:00Z), and writes the profile, edits, logs and abuse tables.
4. `revert_times.py` finds, for edits tagged `mw-reverted`, the revision that reverted them and when (sha1 method).

**Rules**: DATA_PROCESSING R2.2 (how the account was created), R6.1 to R6.3 (identity by user ID), R7.1, R7.6, R7.7, R7.8.

**Paper numbers that depend on M3.** All pre-question covariates and post-question outcomes in M7 (edit counts,
namespaces, reverts, abuse filter hits, account age), the account creation action used by M4 for the sample rule D10,
and the `late_fetch` sensitivity analysis in M8.

**How to use this notebook.** The saved outputs are from the final run; to rerun, go from top to bottom (about 3
minutes). Section 3 recomputes every derived field from the raw files and prints `[OK ]` or `[FAIL]`. Section 5 puts raw records next to processed values for
fixed tracer people and for random people; change `SEED` in the setup cell to draw new random people. Every link
opens the record on Wikipedia so you can check it by eye.

**What this checks.** Nothing yet: this cell loads the four M3 tables and the M2 question list, and draws the random
people used in section 5. It prints the table sizes so you can see the scale of the data.

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────
SEED = 0        # change SEED (any whole number) to draw a fresh random sample in section 5
N_RANDOM = 5    # how many random examples to show per group in section 5

import json, re, sys, time
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np, pandas as pd, pyarrow.dataset as ds, orjson
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import config
from pipeline.common.io import read_jsonl, iter_jsonl
from pipeline.common.tracer import tracer_frame
from checks._show import Checks, compare, pick, oldid, diff, contribs, logid
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
T0 = time.time()
M3, S6, RAW3 = config.INTERIM / "m3", config.INTERIM / "m3" / "s6", config.RAW / "m3"
CUT = config.DATA_CUTOFF_ISO
# All times in these files are text like "2024-01-31T12:00:00Z". Text in this format sorts in time order,
# so "ts < CUT" is a simple and exact cutoff test (section 3 first checks that every time has this format).
ISO = r"^\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ$"

prof = pd.read_parquet(M3 / "mentee_profile.parquet")
P = prof.set_index("mentee_uid")
edits = ds.dataset(M3 / "mentee_edits").to_table(
    columns=["uid", "revid", "parentid", "ns", "timestamp", "size", "sizediff", "tags", "source"]).to_pandas()
logs = pd.read_parquet(M3 / "mentee_logs.parquet")
abuse = pd.read_parquet(M3 / "mentee_abuse.parquet")
q = pd.read_parquet(config.INTERIM / "m2" / "questions.parquet", columns=["question_revid", "mentee_uid", "q_ts"])
q = q[q.mentee_uid.notna()].astype({"mentee_uid": "int64"})
ask = set(q.mentee_uid)
prev_report = json.loads((config.RESULTS / "m3_report.json").read_text()) if (config.RESULTS / "m3_report.json").exists() else None

def clean(v):
    """Turns pandas and numpy values into plain Python values so raw and processed values can be compared."""
    if v is None or v is pd.NA: return None
    if isinstance(v, float) and np.isnan(v): return None
    if isinstance(v, (np.ndarray, list, tuple)): return [str(x) for x in v]
    if isinstance(v, np.bool_): return bool(v)
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return float(v)
    return v

# Random people for section 5 are drawn here, so the large raw files are read only once (in section 3.0).
SAMPLE = {g: list(pick(prof[prof.source == g], N_RANDOM, SEED).mentee_uid) for g in ("s6", "s6+topup", "topup")}
SAMPLE_UIDS = {u for v in SAMPLE.values() for u in v}
ck = Checks()
print(f"askers {len(prof):,} | edits {len(edits):,} | logs {len(logs):,} | abuse filter hits {len(abuse):,}")
print("random people drawn with SEED =", SEED, ":", SAMPLE)

askers 37,095 | edits 3,522,460 | logs 547,492 | abuse filter hits 96,902
random people drawn with SEED = 0 : {'s6': [49614674, 49861173, 47471334, 49884476, 42195541], 's6+topup': [46649515, 48911393, 51240803, 51433711, 49350450], 'topup': [51250241, 49977411, 51484953, 43060146, 45293812]}


**What the output shows.** In the run on the final data (cutoff 2026-04-20T14:00Z) there are 37,095 askers,
3,522,460 edits, 547,492 log entries and 96,902 abuse filter hits. The 15 random people (5 per data source) are
listed by user ID; they are examined one by one in section 5.

### Data dictionary

One table per output. "Checked in" names the notebook cell that verifies the column (3.x are invariants over all rows,
5.x are random raw-versus-processed examples, 4.x are comparisons with independent data). The cell after each table
confirms that the dictionary lists every column in the file.

**What this checks.** The data dictionary below explains every column. The check confirms that the dictionary and
the real files list the same columns, so no column is undocumented.

In [2]:
DICT = {}
DICT["mentee_profile"] = {
    "mentee_uid": ("user ID of the asker (primary key)", "3.1 one row per asker"),
    "source": ("where the data came from: s6, s6+topup (s6 was incomplete, fetched again) or topup (not in s6)", "3.1 recomputed from raw file membership; 5.2 to 5.4"),
    "late_fetch": ("True when the person was fetched after the cutoff (top-up), so later deletions and tags cannot be undone", "3.1"),
    "fetched_at": ("when the profile was fetched (2026-04-23 for s6, the top-up time otherwise)", "3.1; 5.2 to 5.4"),
    "name_at_fetch": ("user name at fetch time", "3.1 against raw s6 profile or raw top-up user; 5.2 to 5.4"),
    "registration": ("registration time from the profile (falls back to the 2026-10 registration file when empty)", "3.1; 4.3 against an independent June fetch; 5.2 to 5.4"),
    "editcount": ("MediaWiki edit counter at fetch time (includes deleted edits)", "3.1; 5.2 to 5.4; 7"),
    "blocked": ("blocked at fetch time", "3.1; 5.2 to 5.4"),
    "groups": ("user groups at fetch time", "3.1; 5.2 to 5.4"),
    "hidden_or_missing": ("the API said the user is hidden or missing (top-up only)", "3.1"),
    "current_name": ("user name in the 2026-10 registration file", "3.1; 5.2 to 5.4"),
    "creation_action": ("action of the account creation log: create = registered themselves; create2/byemail = made by another user; autocreate = came from another wiki", "3.4 recomputed for all; 5.2 to 5.4 raw log record"),
    "creation_ts": ("time of the account creation log entry", "3.4; 5.2 to 5.4; 6 (gap to registration)"),
    "creation_performer": ("user who performed the creation (the person themselves for create)", "3.4; 5.2 to 5.4"),
    "creation_logid": ("log ID of the creation entry (open with the logid link)", "3.4; 5.2 to 5.4"),
    "creation_source": ("log = found in the person's own logs; lookup = found by registration time (creation.jsonl); none = not found", "3.4; 6"),
    "n_edits": ("number of rows of this person in mentee_edits", "3.1 recount; 5.2 to 5.4 recount from raw"),
    "n_logs": ("number of rows of this person in mentee_logs", "3.1 recount; 3.3 from raw; 5.2 to 5.4"),
    "n_abuse": ("number of rows of this person in mentee_abuse", "3.1 recount; 5.2 to 5.4 from raw"),
}
DICT["mentee_edits"] = {
    "uid": ("user ID of the editor", "3.2 (uid, revid) pairs come from the raw files; 5.2 to 5.4"),
    "revid": ("revision ID (unique)", "3.2 unique and equal to the union of raw s6 and raw top-up; 5.2 to 5.4"),
    "parentid": ("revision ID of the page version before this edit", "3.2 all top-up rows vs raw; 5.2 to 5.4 field check"),
    "ns": ("namespace number (0 = article)", "3.2 top-up rows; 5.2 to 5.4"),
    "title": ("page title at fetch time", "3.2 top-up rows; 5.2 to 5.4"),
    "timestamp": ("time of the edit; must be before the cutoff", "3.2 format and cutoff; 5.2 to 5.4"),
    "size": ("page size in bytes after the edit", "3.2 top-up rows; 5.2 to 5.4"),
    "sizediff": ("bytes added (negative = removed)", "3.2 top-up rows; 5.2 to 5.4"),
    "tags": ("change tags at fetch time (for example mw-reverted)", "3.2 top-up rows; 5.2 to 5.4"),
    "minor": ("marked as a minor edit", "3.2 top-up rows; 5.2 to 5.4"),
    "new": ("the edit created the page", "3.2 top-up rows; 5.2 to 5.4"),
    "comment": ("edit summary", "3.2 top-up rows; 5.2 to 5.4"),
    "source": ("s6 or topup (when both have the edit, s6 is kept)", "3.2"),
}
DICT["mentee_logs"] = {
    "uid": ("user ID of the asker the entry belongs to", "3.3 key set equals raw"),
    "logid": ("log ID; (uid, logid) is unique", "3.3"),
    "type": ("log type (newusers, block, rights, ...)", "3.3 every row vs raw"),
    "action": ("log action (create, autocreate, ...)", "3.3 every row vs raw"),
    "timestamp": ("time of the log entry; before the cutoff", "3.3 every row vs raw"),
    "ns": ("namespace of the target page", "3.3 every row vs raw"),
    "title": ("target page", "3.3 every row vs raw"),
    "performer": ("user who performed the action", "3.3 every row vs raw"),
    "params": ("log parameters as JSON text (newusers entries hold the new account's userid)", "3.3 every row vs raw"),
    "comment": ("log comment", "3.3 every row vs raw"),
    "source": ("s6 or topup (when both have it, s6 is kept)", "3.3"),
}
DICT["mentee_abuse"] = {
    "uid": ("user ID of the asker", "3.3 key set"),
    "abuse_id": ("abuse log ID; (uid, abuse_id) is unique", "3.3"),
    "filter_id": ("ID of the filter that was hit", "3.3 top-up rows vs raw; 5.2 to 5.4"),
    "filter": ("name of the filter", "3.3 top-up rows; 5.2 to 5.4"),
    "action": ("what the user tried to do (edit, createaccount, ...)", "3.3 top-up rows; 5.2 to 5.4"),
    "result": ("what the filter did (warn, disallow, tag, empty = log only)", "3.3 top-up rows; 5.2 to 5.4"),
    "timestamp": ("time of the hit; before the cutoff", "3.3; 5.2 to 5.4"),
    "ns": ("namespace of the page", "3.3 top-up rows; 5.2 to 5.4"),
    "title": ("page", "3.3 top-up rows; 5.2 to 5.4"),
    "source": ("s6 or topup", "3.3"),
}
DICT["revert_times (raw jsonl)"] = {
    "revid": ("the edit tagged mw-reverted", "3.5 is an mw-reverted edit of the same person"),
    "uid": ("its editor", "3.5"),
    "ts": ("time of the edit", "3.5 equals the edit time"),
    "fetched_at": ("when the lookup was made", "3.5 after the cutoff"),
    "method": ("sha1 = page returned to an earlier version; undo_comment = an undo naming the edit; page_deleted; not_found; api_error", "3.5; 6"),
    "revert_revid": ("revision that reverted the edit", "3.5 later than the edit; 5.5 links"),
    "revert_ts": ("time of the revert", "3.5 after the edit; 5.5"),
    "lag_hours": ("hours from the edit to the revert", "3.5 recomputed; 5.5"),
    "scanned": ("for not_found: how many later revisions were checked", "6"),
    "error": ("for api_error: the error text", "6"),
}
def show_dict(name, cols):
    t = pd.DataFrame([{"column": k, "meaning": v[0], "checked in": v[1]} for k, v in DICT[name].items()])
    print(f"\n{name}"); print(t.to_string(index=False, max_colwidth=110))
    if name.startswith("revert_times"):   # optional fields (scanned, error) appear only once such a result exists
        ck.add(f"data dictionary of {name} lists every field in the file", set(cols) <= set(DICT[name]))
    else:
        ck.add(f"data dictionary of {name} lists exactly the columns in the file", set(DICT[name]) == set(cols))
show_dict("mentee_profile", prof.columns)
show_dict("mentee_edits", ds.dataset(M3 / "mentee_edits").schema.names)
show_dict("mentee_logs", logs.columns)
show_dict("mentee_abuse", abuse.columns)
_rt_cols = set().union(*[set(r) for p in RAW3.glob("revert_times*.jsonl") for r in read_jsonl(p)[:5000]])
show_dict("revert_times (raw jsonl)", _rt_cols)


mentee_profile
            column                                                                                                        meaning                                                checked in
        mentee_uid                                                                             user ID of the asker (primary key)                                     3.1 one row per asker
            source                 where the data came from: s6, s6+topup (s6 was incomplete, fetched again) or topup (not in s6)       3.1 recomputed from raw file membership; 5.2 to 5.4
        late_fetch       True when the person was fetched after the cutoff (top-up), so later deletions and tags cannot be undone                                                       3.1
        fetched_at                                    when the profile was fetched (2026-04-23 for s6, the top-up time otherwise)                                           3.1; 5.2 to 5.4
     name_at_fetch                          


revert_times (raw jsonl)
      column                                                                                                        meaning                                    checked in
       revid                                                                                    the edit tagged mw-reverted 3.5 is an mw-reverted edit of the same person
         uid                                                                                                     its editor                                           3.5
          ts                                                                                               time of the edit                      3.5 equals the edit time
  fetched_at                                                                                       when the lookup was made                          3.5 after the cutoff
      method sha1 = page returned to an earlier version; undo_comment = an undo naming the edit; page_deleted; not_found... 

**What the output shows.** All five dictionary checks print `[OK ]`: every column in every output file is listed and
explained, and the dictionary names no column that does not exist.

## 2. Files and manifests

**What this checks.** Each step leaves a small receipt (a "manifest") saying when it ran, which cutoff it used and
how many rows it wrote. If one step used a different cutoff, or the tables were changed after the receipt was
written, the numbers in the paper could come from mismatched data.

Each step writes a manifest (finish time, cutoff, input and output files, counts). The checks below confirm that
the manifests use the same cutoff as `config.py` and that the counts in the M3 manifest equal the tables on disk.

In [3]:
mans = {}
for name in ("m3_index", "m3_collect", "m3"):
    p = config.INTERIM / name / "manifest.json"
    man = mans[name] = json.loads(p.read_text())
    print(f"\n[{name}] finished {man['finished_at']} | cutoff {man['data_cutoff']} | {man['elapsed_sec']} s")
    display(pd.DataFrame(man["inputs"] + man["outputs"])[[c for c in ("path", "exists", "bytes", "mtime") if c in pd.DataFrame(man["inputs"] + man["outputs"]).columns]])
    print(json.dumps(man["counts"], ensure_ascii=False)[:900])
    ck.add(f"{name} manifest uses the cutoff in config.py", man["data_cutoff"] == CUT)
c3 = mans["m3"]["counts"]
ck.add("m3 manifest counts equal the tables on disk (askers, edits, logs, abuse)",
       (c3["askers"], c3["edits"], c3["logs"], c3["abuse"]) == (len(prof), len(edits), len(logs), len(abuse)))
ck.add("the merge (m3) ran after both m3_index and m3_collect",
       max(mans["m3_index"]["finished_at"], mans["m3_collect"]["finished_at"]) <= mans["m3"]["finished_at"])
# collect_topup used coverage.json from m3_index; if m3_index ran again later, its list of askers missing from s6 must be unchanged
cov = json.loads((S6 / "coverage.json").read_text())
ck.add("askers missing from s6 in coverage.json = the askers with source 'topup'",
       set(cov["missing_uids"]) == set(prof[prof.source == "topup"].mentee_uid))
print("\nraw files of M3 (data/raw/m3):")
for p in sorted(RAW3.iterdir()):
    print(f"  {p.name:28s} {p.stat().st_size / 1e6:9.1f} MB  modified {pd.Timestamp(p.stat().st_mtime, unit='s'):%Y-%m-%d %H:%M}")


[m3_index] finished 2026-10-24T21:30:10.326198+00:00 | cutoff 2026-04-20T14:00:00Z | 72.4 s


,path,exists,bytes,mtime
0,../0wiki-mentor-github/s6_user_profiles.jsonl,True,14728086,2026-04-23T15:08:29.830508+00:00
1,../0wiki-mentor-github/s6_user_contribs.jsonl,True,3447496466,2026-04-23T17:13:44.463447+00:00
2,../0wiki-mentor-github/s6_user_logevents.j...,True,111091255,2026-04-24T04:08:58.755770+00:00
3,../0wiki-mentor-github/s6_user_abuselog.jsonl,True,5132370172,2026-04-24T04:47:43.560181+00:00
4,./data/interim/m...,True,13522465,2026-10-24T21:29:47.816375+00:00
5,./data/interim/m...,True,2419554,2026-10-24T21:30:08.313116+00:00


{"askers": 37095, "in_profiles": 35732, "missing": 1363, "identity_failed": 0, "edits": {"users": 35732, "edits": 3227855, "dropped_after_cutoff": 6860, "edit_count_mismatch": 0, "users_with_edits": 35732}, "logs": {"users": 35732, "events": 215895, "dropped_after_cutoff": 13}, "abuse": {"users": 35732, "entries": 77585, "dropped_after_cutoff": 101}}
  [OK ] m3_index manifest uses the cutoff in config.py

[m3_collect] finished 2026-10-24T21:28:51.230142+00:00 | cutoff 2026-04-20T14:00:00Z | 169.7 s


,path,exists,bytes,mtime
0,./data/interim/m...,True,18429,2026-10-24T18:29:30.835967+00:00
1,./data/raw/m4_re...,True,4430402,2026-10-07T04:51:24.868585+00:00
2,./data/raw/m3/to...,True,57873790,2026-10-24T21:28:41.523429+00:00
3,./data/raw/m3/cr...,True,587548,2026-10-24T21:28:51.229017+00:00


{"topup": 1527, "creation_lookups": 1610, "creation_found_now": 4, "api": {"cache_hit": 0, "network": 164, "retries": 0}}
  [OK ] m3_collect manifest uses the cutoff in config.py

[m3] finished 2026-10-26T14:05:04.080471+00:00 | cutoff 2026-04-20T14:00:00Z | 34.1 s


,path,exists,bytes,mtime
0,./data/interim/m...,True,16521,2026-10-24T21:30:10.321794+00:00
1,./data/raw/m3/to...,True,57873790,2026-10-24T21:28:41.523429+00:00
2,./data/raw/m3/cr...,True,587548,2026-10-24T21:28:51.229017+00:00
3,./data/interim/m...,True,3324927,2026-10-26T14:05:03.463961+00:00
4,./data/interim/m...,True,27097194,2026-10-26T14:05:03.995639+00:00
5,./data/interim/m...,True,2904315,2026-10-26T14:05:04.073846+00:00


{"askers": 37095, "source": {"s6": 33642, "s6+topup": 2090, "topup": 1363}, "creation_action": {"create": 36335, "byemail": 367, "autocreate": 225, "create2": 85, "null": 82, "forcecreatelocal": 1}, "creation_source": {"log": 37012, "none": 82, "lookup": 1}, "edits": 3522460, "logs": 547492, "abuse": 96902, "askers_without_edits": 0}
  [OK ] m3 manifest uses the cutoff in config.py
  [OK ] m3 manifest counts equal the tables on disk (askers, edits, logs, abuse)
  [OK ] the merge (m3) ran after both m3_index and m3_collect
  [OK ] askers missing from s6 in coverage.json = the askers with source 'topup'

raw files of M3 (data/raw/m3):
  creation.jsonl                     0.6 MB  modified 2026-10-24 21:28
  revert_times.jsonl                 8.2 MB  modified 2026-10-26 09:35
  revert_times_sample.jsonl          0.3 MB  modified 2026-10-24 23:08
  stale.json                         0.1 MB  modified 2026-10-24 18:56
  stale_edits.jsonl                 76.5 MB  modified 2026-10-24 18:56
  to

**What the output shows.** All three steps used the cutoff 2026-04-20T14:00:00Z, the merge ran last (2026-10-26)
after the other two (2026-10-24), and the counts in the merge manifest equal the tables on disk. The 1,363 askers that
s6 lacks are exactly the 1,363 "topup" people. The s6 index dropped 6,860 edits, 13 log entries and 101 abuse filter
hits that fell on or after the cutoff (s6 was downloaded three days after it). All six checks pass.

## 3. Invariants (all should be OK)

### 3.0 Read the raw files once

**What this checks.** This cell only reads the raw downloads into memory, so the later checks can compare the
processed tables against the original records instead of trusting the pipeline's own intermediate files.

Everything in section 3 is recomputed from the **raw** files, not from the intermediate s6 index, where possible:
the raw s6 profile, edit, log and abuse files (stored by user name), and the raw top-up files. For the two very large
s6 files (edits 3.4 GB, abuse 5.1 GB) we read only the counter at the start of each line, plus the full line for the
random people of section 5. This takes about 30 to 60 seconds.

In [4]:
t = time.time()
HEAD = re.compile(rb'^\{"username":\s*"((?:[^"\\]|\\.)*)"')
def head_name(line):
    m = HEAD.match(line)
    return json.loads(b'"' + m.group(1) + b'"') if m else None

# raw s6 profiles: user name at fetch time -> profile record (only records that have a user ID)
s6_prof_raw = {}
for line in open(config.EXTERNAL["s6_profiles"], "rb"):
    r = orjson.loads(line)
    if r.get("userid") is not None:
        s6_prof_raw[r["username"]] = r
s6_name = {r["userid"]: n for n, r in s6_prof_raw.items() if r["userid"] in ask}     # asker uid -> s6 name
name2uid = {n: u for u, n in s6_name.items()}
ck.add("no asker has two different names in the raw s6 profiles", len(name2uid) == len(s6_name))

# raw s6 edits: the edit_count field at the start of each line; the full line only for sampled people
EC = re.compile(rb'"edit_count":\s*"?(\d+)')
s6_edit_count, s6_full = {}, {}
for line in open(config.EXTERNAL["s6_contribs"], "rb"):
    u = name2uid.get(head_name(line))
    if u is None: continue
    m = EC.search(line[:3000]); s6_edit_count[u] = int(m.group(1)) if m else None
    if u in SAMPLE_UIDS: s6_full[u] = orjson.loads(line)["edits"]

# raw s6 logs: small enough to parse every asker line
s6_log_rows = []
for line in open(config.EXTERNAL["s6_logevents"], "rb"):
    u = name2uid.get(head_name(line))
    if u is None: continue
    for e in orjson.loads(line).get("events") or []:
        s6_log_rows.append((u, e.get("logid"), e.get("type"), e.get("action"), e.get("timestamp"), e.get("ns"), e.get("title"),
                            e.get("user"), e.get("params") or {}, e.get("comment"), "s6"))

# raw s6 abuse filter hits: the abuse_count field; the full line only for sampled people
AC = re.compile(rb'"abuse_count":\s*"?(\d+)')
s6_abuse_count, s6_abuse_full = {}, {}
for line in open(config.EXTERNAL["s6_abuselog"], "rb"):
    u = name2uid.get(head_name(line))
    if u is None: continue
    m = AC.search(line[:3000]); s6_abuse_count[u] = int(m.group(1)) if m else None
    if u in SAMPLE_UIDS: s6_abuse_full[u] = orjson.loads(line).get("entries") or []

# raw top-up: topup.jsonl, then topup_stale.jsonl; a later record for the same uid replaces an earlier one (as in merge_activity)
topup, n_topup_lines = {}, Counter()
for fname in ("topup.jsonl", "topup_stale.jsonl"):
    for line in open(RAW3 / fname, "rb"):
        r = orjson.loads(line); n_topup_lines[fname] += 1
        if r["uid"] in ask:
            topup[r["uid"]] = dict(r, _file=fname)
creation_raw = {}
for r in iter_jsonl(RAW3 / "creation.jsonl"):
    creation_raw.setdefault(r["uid"], []).append(r)
stale = {int(k): v for k, v in json.loads((RAW3 / "stale.json").read_text()).items()}
stale_edits = {r["uid"]: r["edits"] for r in iter_jsonl(RAW3 / "stale_edits.jsonl")}
reg_now = json.loads(config.RAW_FILES["registration"].read_text())
print(f"raw s6: {len(s6_name):,} askers in profiles, {len(s6_edit_count):,} edit lines, {len(s6_log_rows):,} log events, "
      f"{len(s6_abuse_count):,} abuse lines | top-up lines {dict(n_topup_lines)}, askers {len(topup):,} | "
      f"creation.jsonl {len(creation_raw):,} uids | stale.json {len(stale):,} ({time.time() - t:.0f} s)")

  [OK ] no asker has two different names in the raw s6 profiles


raw s6: 35,732 askers in profiles, 35,732 edit lines, 254,452 log events, 35,732 abuse lines | top-up lines {'topup.jsonl': 1527, 'topup_stale.jsonl': 2085}, askers 3,453 | creation.jsonl 1,610 uids | stale.json 2,090 (26 s)


**What the output shows.** The raw s6 files hold 35,732 askers (all with exactly one name, check `[OK ]`). The top-up
files hold 1,527 + 2,085 person records, of which 3,453 are current askers (1,363 topup + 2,090 s6+topup). The other
159 lines in `topup.jsonl` are people who are not in the current M2 asker list; they are ignored, which is fine.

### 3.1 Profile

**What this checks.** For every one of the 37,095 askers, each profile field (name, registration time, edit counter,
blocked, groups, and so on) is rebuilt from the raw download with separate code and compared with the stored value.
Registration time feeds the account age covariate, so it has to be right for everyone.

`source` is recomputed from which raw files contain the person; the profile fields are recomputed from the raw s6
profile line (s6 people) or the raw top-up `user` record (top-up people), with the documented fallback to the
2026-10 registration file when the registration time is empty.

In [5]:
def expected_profile(uid):
    """Profile fields of one asker, built directly from the raw records."""
    now = reg_now.get(str(uid)) or {}
    if uid in topup:
        u = topup[uid]["user"]
        return {"source": "s6+topup" if uid in s6_name else "topup", "late_fetch": True, "fetched_at": topup[uid]["fetched_at"],
                "name_at_fetch": u.get("name"), "registration": u.get("registration") or now.get("registration"),
                "editcount": u.get("editcount"), "blocked": "blockid" in u, "groups": list(u.get("groups") or []),
                "hidden_or_missing": bool(u.get("missing") or u.get("hidden")), "current_name": now.get("name")}
    if uid in s6_name:
        r = s6_prof_raw[s6_name[uid]]
        return {"source": "s6", "late_fetch": False, "fetched_at": "2026-04-23", "name_at_fetch": r["username"],
                "registration": r.get("registration") or now.get("registration"), "editcount": r.get("editcount"),
                "blocked": bool(r.get("blocked")), "groups": list(r.get("groups") or []), "hidden_or_missing": False,
                "current_name": now.get("name")}
    return {"source": "none", "current_name": now.get("name")}

exp = {u: expected_profile(u) for u in P.index}
ck.add("one profile row per asker, and the set of uids equals the M2 askers", prof.mentee_uid.is_unique and set(prof.mentee_uid) == ask)
ck.add("source is s6, s6+topup or topup for everyone (no asker without data)", prof.source.isin(["s6", "s6+topup", "topup"]).all())
bad_fields = {}
for f in ("source", "late_fetch", "fetched_at", "name_at_fetch", "registration", "editcount", "blocked", "groups",
          "hidden_or_missing", "current_name"):
    bad = [u for u in P.index if f in exp[u] and clean(P.at[u, f]) != clean(exp[u][f])]
    bad_fields[f] = bad
    ck.add(f"profile.{f} equals the raw value for every asker (mismatches: {len(bad)})", not bad)
ck.add("late_fetch is True exactly for s6+topup and topup", (prof.late_fetch == (prof.source != "s6")).all())
for col, tbl in (("n_edits", edits), ("n_logs", logs), ("n_abuse", abuse)):
    cnt = tbl.groupby("uid").size()
    ck.add(f"profile.{col} equals the number of rows of the person in its table", (P[col] == cnt.reindex(P.index).fillna(0).astype(int)).all())
for f, bad in bad_fields.items():
    if bad:
        print(f"\n{f}: first mismatches"); print([(u, clean(P.at[u, f]), exp[u].get(f)) for u in bad[:5]])

  [OK ] one profile row per asker, and the set of uids equals the M2 askers
  [OK ] source is s6, s6+topup or topup for everyone (no asker without data)


  [OK ] profile.source equals the raw value for every asker (mismatches: 0)
  [OK ] profile.late_fetch equals the raw value for every asker (mismatches: 0)


  [OK ] profile.fetched_at equals the raw value for every asker (mismatches: 0)
  [OK ] profile.name_at_fetch equals the raw value for every asker (mismatches: 0)


  [OK ] profile.registration equals the raw value for every asker (mismatches: 0)


  [OK ] profile.editcount equals the raw value for every asker (mismatches: 0)
  [OK ] profile.blocked equals the raw value for every asker (mismatches: 0)


  [OK ] profile.groups equals the raw value for every asker (mismatches: 0)
  [OK ] profile.hidden_or_missing equals the raw value for every asker (mismatches: 0)


  [OK ] profile.current_name equals the raw value for every asker (mismatches: 0)
  [OK ] late_fetch is True exactly for s6+topup and topup
  [OK ] profile.n_edits equals the number of rows of the person in its table
  [OK ] profile.n_logs equals the number of rows of the person in its table
  [OK ] profile.n_abuse equals the number of rows of the person in its table


**What the output shows.** Every field matches the raw value for every asker (0 mismatches in each of 10 fields), no
asker is left without data, and the edit, log and abuse counts in the profile equal the rows in the tables.

### 3.2 Edits

**What this checks.** The edit table is the base of almost every covariate and outcome. The checks make sure that no
edit is lost or added during merging, that no edit is counted twice, that nothing on or after the cutoff gets in, and
that each newcomer's question edit (from M2) is in their own history, which shows the user IDs line up across modules.

The merged edit table must be exactly the union of the s6 edits and the raw top-up edits before the cutoff, with
each revision once. For top-up rows every field is compared with the raw API record. For s6 rows we can only check
the counter at the start of each raw line here (the full field check of s6 rows is done for the random people in 5.2).

In [6]:
t = time.time()
ck.add("every edit time has the ISO format YYYY-MM-DDTHH:MM:SSZ", edits.timestamp.str.match(ISO).all())
ck.add("no edit on or after the cutoff", (edits.timestamp < CUT).all())
ck.add("revid is unique (duplicates removed)", edits.revid.is_unique)
ck.add("every editor is an asker", edits.uid.isin(ask).all())
ck.add("source is s6 or topup", edits.source.isin(["s6", "topup"]).all())

s6idx = ds.dataset(S6 / "edits").to_table(columns=["uid", "revid", "timestamp"]).to_pandas()
tp_rows = [(u, e.get("revid"), e.get("parentid"), e.get("ns"), e.get("title"), e.get("timestamp"), e.get("size"), e.get("sizediff"),
            list(e.get("tags") or []), bool(e.get("minor")), bool(e.get("new")), e.get("comment"))
           for u, r in topup.items() for e in r.get("contribs") or []]
tpe = pd.DataFrame(tp_rows, columns=["uid", "revid", "parentid", "ns", "title", "timestamp", "size", "sizediff", "tags", "minor", "new", "comment"])
tpe_in = tpe[tpe.timestamp < CUT]
s6set, tpset, merged = set(s6idx.revid), set(tpe_in.revid), set(edits.revid)
print(f"s6 index edits {len(s6set):,} | raw top-up edits {len(tpe):,} (on or after cutoff, dropped: {len(tpe) - len(tpe_in):,}) "
      f"| top-up edits already in s6 {len(tpset & s6set):,} | merged {len(merged):,}")
ck.add("merged revids = s6 revids + raw top-up revids before the cutoff (nothing lost, nothing extra)", merged == s6set | tpset)
ck.add("no raw top-up edit on or after the cutoff got into the table", not (set(tpe[tpe.timestamp >= CUT].revid) & merged))
ck.add("source is s6 exactly when the revid is in the s6 index", ((edits.source == "s6") == edits.revid.isin(s6set)).all())
pairs = set(zip(s6idx.uid, s6idx.revid)) | set(zip(tpe_in.uid, tpe_in.revid))
ck.add("every (uid, revid) pair in the table is a pair from the raw files", set(zip(edits.uid, edits.revid)) <= pairs)

# s6 index against the raw s6 lines: per person, the index has at most the raw edit_count (fewer only because of the cutoff)
n_idx = s6idx.groupby("uid").size()
raw_ec = pd.Series(s6_edit_count)
ck.add("every asker in the raw s6 edit file has edit_count >= edits in the s6 index", (n_idx.reindex(raw_ec.index).fillna(0) <= raw_ec).all())
dropped = mans["m3_index"]["counts"]["edits"]["dropped_after_cutoff"]
ck.add(f"sum of raw s6 edit_count ({int(raw_ec.sum()):,}) = s6 index edits ({len(s6idx):,}) + dropped after cutoff ({dropped:,})",
       int(raw_ec.sum()) == len(s6idx) + dropped)

# full field check of every top-up row against the raw API record
mt = ds.dataset(M3 / "mentee_edits").to_table(filter=ds.field("source") == "topup").to_pandas()
j = mt.merge(tpe_in.drop_duplicates("revid"), on="revid", suffixes=("", "_raw"))
ck.add("every top-up row in the table has its raw record", len(j) == len(mt))
for f in ("uid", "parentid", "ns", "title", "timestamp", "size", "sizediff", "tags", "minor", "new", "comment"):
    n_bad = sum(clean(a) != clean(b) for a, b in zip(j[f], j[f + "_raw"]))
    ck.add(f"top-up rows: {f} equals the raw API value ({n_bad} differ)", n_bad == 0)

# stale people: the edits added for them are exactly the ones find_stale found, and they are newer than their last s6 edit
st_tp = {u: r for u, r in topup.items() if r["_file"] == "topup_stale.jsonl"}
ck.add("for every stale person, top-up edits = the edits in stale_edits.jsonl",
       all({e["revid"] for e in r["contribs"]} == {e["revid"] for e in stale_edits.get(u, [])} for u, r in st_tp.items()))
last_s6 = s6idx.groupby("uid").timestamp.max()
newer = tpe_in[tpe_in.uid.isin([u for u, v in stale.items() if v["reason"] == "newer_edits"])]
ck.add("edits added for 'newer_edits' people are all later than their last s6 edit", (newer.timestamp > newer.uid.map(last_s6)).all())
ck.add("every stale.json person is an s6+topup asker", all(P.at[u, "source"] == "s6+topup" for u in stale if u in P.index))

# identity and completeness (R6.1, R7.6)
first_edit = edits.groupby("uid").timestamp.min()
reg = P.registration.dropna()
early = reg.index[reg.values > first_edit.reindex(reg.index).fillna("9999").values]
ck.add(f"registration is not later than the first edit ({len(early)} askers fail)", len(early) == 0)
ck.add("every question revision of M2 is in its asker's edits (R7.6)",
       set(zip(q.mentee_uid, q.question_revid)) <= set(zip(edits.uid, edits.revid)))
ck.add("every asker has at least one edit (the question itself is an edit)", (prof.n_edits > 0).all())
print(f"({time.time() - t:.0f} s)")

  [OK ] every edit time has the ISO format YYYY-MM-DDTHH:MM:SSZ
  [OK ] no edit on or after the cutoff


  [OK ] revid is unique (duplicates removed)
  [OK ] every editor is an asker
  [OK ] source is s6 or topup


s6 index edits 3,227,855 | raw top-up edits 304,547 (on or after cutoff, dropped: 7,491) | top-up edits already in s6 2,451 | merged 3,522,460
  [OK ] merged revids = s6 revids + raw top-up revids before the cutoff (nothing lost, nothing extra)
  [OK ] no raw top-up edit on or after the cutoff got into the table


  [OK ] source is s6 exactly when the revid is in the s6 index


  [OK ] every (uid, revid) pair in the table is a pair from the raw files
  [OK ] every asker in the raw s6 edit file has edit_count >= edits in the s6 index
  [OK ] sum of raw s6 edit_count (3,234,715) = s6 index edits (3,227,855) + dropped after cutoff (6,860)


...

  [OK ] top-up rows: new equals the raw API value (0 differ)


  [OK ] top-up rows: comment equals the raw API value (0 differ)
  [OK ] for every stale person, top-up edits = the edits in stale_edits.jsonl


  [OK ] edits added for 'newer_edits' people are all later than their last s6 edit
  [OK ] every stale.json person is an s6+topup asker


  [OK ] registration is not later than the first edit (0 askers fail)


  [OK ] every question revision of M2 is in its asker's edits (R7.6)
  [OK ] every asker has at least one edit (the question itself is an edit)
(30 s)


**What the output shows.** All 29 edit checks pass. The merged table (3,522,460 edits) is exactly the 3,227,855 s6
edits plus the top-up edits before the cutoff; 7,491 top-up edits on or after the cutoff were dropped, and 2,451
top-up edits that s6 already had were kept once. For the 304,547 top-up edits every field equals the raw API record.
The raw s6 edit counts add up exactly (3,234,715 = 3,227,855 kept + 6,860 after the cutoff). No one's registration is
later than their first edit, and every M2 question is found in its asker's edits.

### 3.3 Logs and abuse filter hits

**What this checks.** Log entries (for example "account created" or "blocked") and abuse filter hits (an automatic
rule on Wikipedia flagged an action, such as adding a suspicious link) are used as covariates. The checks compare the
stored rows with the raw downloads, the same way as for edits.

Logs: the raw s6 log file is small, so **every** log row (s6 and top-up) is compared with its raw record, field by
field. Abuse filter hits: the key set is checked for everyone, every top-up row is compared with the raw record, and
s6 rows are compared with the raw file for the random people in 5.2.

In [7]:
t = time.time()
LC = ["uid", "logid", "type", "action", "timestamp", "ns", "title", "performer", "params", "comment", "source"]
tp_log_rows = [(u, e.get("logid"), e.get("type"), e.get("action"), e.get("timestamp"), e.get("ns"), e.get("title"), e.get("user"),
                e.get("params") or {}, e.get("comment"), "topup") for u, r in topup.items() for e in r.get("logs") or []]
rawlogs = pd.DataFrame(s6_log_rows + tp_log_rows, columns=LC)
rawlogs = rawlogs[rawlogs.timestamp < CUT].drop_duplicates(["uid", "logid"], keep="first")   # s6 rows come first, so s6 wins
ck.add("every log time has the ISO format", logs.timestamp.str.match(ISO).all())
ck.add("no log entry on or after the cutoff", (logs.timestamp < CUT).all())
ck.add("(uid, logid) is unique in mentee_logs", not logs.duplicated(["uid", "logid"]).any())
ck.add("log keys (uid, logid) = raw s6 keys + raw top-up keys before the cutoff",
       set(zip(logs.uid, logs.logid.astype(int))) == set(zip(rawlogs.uid, rawlogs.logid)))
jl = logs.assign(logid=logs.logid.astype(int)).merge(rawlogs, on=["uid", "logid"], suffixes=("", "_raw"))
for f in ("type", "action", "timestamp", "ns", "title", "performer", "comment", "source"):
    n_bad = sum(clean(a) != clean(b) for a, b in zip(jl[f], jl[f + "_raw"]))
    ck.add(f"logs: {f} equals the raw value in every row ({n_bad} differ)", n_bad == 0)
n_bad = sum(json.loads(a or "{}") != b for a, b in zip(jl.params, jl.params_raw))
ck.add(f"logs: params (parsed JSON) equals the raw params in every row ({n_bad} differ)", n_bad == 0)
ck.add("profile.n_logs equals the count recomputed from raw logs",
       (P.n_logs == rawlogs.groupby("uid").size().reindex(P.index).fillna(0).astype(int)).all())

s6ab = pd.read_parquet(S6 / "abuse.parquet")
AB = ["uid", "abuse_id", "filter_id", "filter", "action", "result", "timestamp", "ns", "title"]
tpa = pd.DataFrame([(u, e.get("id"), str(e.get("filter_id") or ""), e.get("filter"), e.get("action"), e.get("result"), e.get("timestamp"),
                     e.get("ns"), e.get("title")) for u, r in topup.items() for e in r.get("abuse") or []], columns=AB)
tpa = tpa[tpa.timestamp < CUT]
ck.add("every abuse filter time has the ISO format", abuse.timestamp.str.match(ISO).all())
ck.add("no abuse filter hit on or after the cutoff", (abuse.timestamp < CUT).all())
ck.add("(uid, abuse_id) is unique in mentee_abuse", not abuse.duplicated(["uid", "abuse_id"]).any())
ck.add("abuse keys = s6 index keys + raw top-up keys before the cutoff",
       set(zip(abuse.uid, abuse.abuse_id.astype(int))) == set(zip(s6ab.uid, s6ab.abuse_id)) | set(zip(tpa.uid, tpa.abuse_id)))
ja = abuse[abuse.source == "topup"].assign(abuse_id=lambda d: d.abuse_id.astype(int)).merge(tpa.drop_duplicates(["uid", "abuse_id"]), on=["uid", "abuse_id"], suffixes=("", "_raw"))
ck.add("every top-up abuse row has its raw record", len(ja) == int((abuse.source == "topup").sum()))
for f in AB[2:]:
    n_bad = sum(clean(a) != clean(b) for a, b in zip(ja[f], ja[f + "_raw"]))
    ck.add(f"abuse (top-up rows): {f} equals the raw value ({n_bad} differ)", n_bad == 0)
n_ab_idx = s6ab.groupby("uid").size()
raw_ac = pd.Series(s6_abuse_count)
ck.add("every asker in the raw s6 abuse file has abuse_count >= hits in the s6 index", (n_ab_idx.reindex(raw_ac.index).fillna(0) <= raw_ac).all())
print(f"raw s6 abuse_count total {int(raw_ac.sum()):,} | s6 index {len(s6ab):,} | dropped after cutoff (manifest) "
      f"{mans['m3_index']['counts']['abuse']['dropped_after_cutoff']} ({time.time() - t:.0f} s)")

  [OK ] every log time has the ISO format
  [OK ] no log entry on or after the cutoff
  [OK ] (uid, logid) is unique in mentee_logs


  [OK ] log keys (uid, logid) = raw s6 keys + raw top-up keys before the cutoff


  [OK ] logs: type equals the raw value in every row (0 differ)


  [OK ] logs: action equals the raw value in every row (0 differ)


  [OK ] logs: timestamp equals the raw value in every row (0 differ)


...

  [OK ] logs: performer equals the raw value in every row (0 differ)


  [OK ] logs: comment equals the raw value in every row (0 differ)


  [OK ] logs: source equals the raw value in every row (0 differ)


  [OK ] logs: params (parsed JSON) equals the raw params in every row (0 differ)
  [OK ] profile.n_logs equals the count recomputed from raw logs
  [OK ] every abuse filter time has the ISO format
  [OK ] no abuse filter hit on or after the cutoff
  [OK ] (uid, abuse_id) is unique in mentee_abuse


  [OK ] abuse keys = s6 index keys + raw top-up keys before the cutoff
  [OK ] every top-up abuse row has its raw record
  [OK ] abuse (top-up rows): filter_id equals the raw value (0 differ)
  [OK ] abuse (top-up rows): filter equals the raw value (0 differ)
  [OK ] abuse (top-up rows): action equals the raw value (0 differ)
  [OK ] abuse (top-up rows): result equals the raw value (0 differ)
  [OK ] abuse (top-up rows): timestamp equals the raw value (0 differ)
  [OK ] abuse (top-up rows): ns equals the raw value (0 differ)
  [OK ] abuse (top-up rows): title equals the raw value (0 differ)
  [OK ] every asker in the raw s6 abuse file has abuse_count >= hits in the s6 index
raw s6 abuse_count total 77,686 | s6 index 77,585 | dropped after cutoff (manifest) 101 (13 s)


**What the output shows.** All 27 checks pass. Every one of the 547,492 log rows matches its raw record in every field
(0 differences). For abuse filter hits, the raw s6 total 77,686 = 77,585 kept + 101 on or after the cutoff, and every
top-up row matches the raw record.

### 3.4 Account creation log (R2.2, R7.7)

**What this checks.** The sample rule in M4 keeps people who registered their own account (`create`), so we must know
how each account was made. The creation entry is recomputed with separate simple code and compared with the profile.
It also checks that the creation entry happens within a few seconds of the registration time, which shows we picked
the right entry.

Recomputed here with simple separate code: among the person's own logs, take the `newusers` entries whose
`params.userid` is the person's uid, and keep the earliest. If there is none, use the first entry with a record in
`creation.jsonl` (found by registration time). Otherwise there is no creation log.

In [8]:
own = {}
for r in logs[logs.type == "newusers"].sort_values(["timestamp", "logid"]).itertuples():
    if str(json.loads(r.params or "{}").get("userid")) == str(r.uid) and r.uid not in own:
        own[r.uid] = {"creation_action": r.action, "creation_ts": r.timestamp, "creation_performer": r.performer,
                      "creation_logid": int(r.logid), "creation_source": "log"}
def expected_creation(uid):
    if uid in own: return own[uid]
    for r in creation_raw.get(uid, []):
        if r.get("event"):
            e = r["event"]
            return {"creation_action": e.get("action"), "creation_ts": e.get("timestamp"), "creation_performer": e.get("user"),
                    "creation_logid": e.get("logid"), "creation_source": "lookup"}
    return {"creation_action": None, "creation_ts": None, "creation_performer": None, "creation_logid": None, "creation_source": "none"}
expc = {u: expected_creation(u) for u in P.index}
for f in ("creation_action", "creation_ts", "creation_performer", "creation_logid", "creation_source"):
    bad = [u for u in P.index if clean(P.at[u, f]) != clean(expc[u][f])]
    ck.add(f"profile.{f} equals the recomputed creation log for every asker (mismatches: {len(bad)})", not bad)
    if bad: print("   first mismatches:", [(u, clean(P.at[u, f]), expc[u][f]) for u in bad[:5]])
ck.add("creation_action is one of create, create2, byemail, autocreate, forcecreatelocal or empty",
       prof.creation_action.dropna().isin(["create", "create2", "byemail", "autocreate", "forcecreatelocal"]).all())
ck.add("every asker with no creation log was looked up in creation.jsonl and not found",
       all(u in creation_raw and not any(r.get("event") for r in creation_raw[u]) for u in prof[prof.creation_source == "none"].mentee_uid))
cr = prof[prof.creation_action == "create"]
same = (cr.creation_performer == cr.name_at_fetch) | (cr.creation_performer == cr.current_name)
print(f"'create' entries whose performer is the person's own name at fetch time or now: {same.mean():.2%} "
      f"(the rest are mostly renames: the log keeps the name used at registration)")
gap = (pd.to_datetime(prof.creation_ts, utc=True) - pd.to_datetime(prof.registration, utc=True)).dt.total_seconds()
prof["creation_gap_s"] = gap
print("seconds from registration to the creation log entry:", gap.describe().round(1).to_dict())
ck.add("for 99% of askers the creation log is within 2 s before to 10 s after registration (R7.7)", gap.between(-2, 10).mean() > .99)

  [OK ] profile.creation_action equals the recomputed creation log for every asker (mismatches: 0)
  [OK ] profile.creation_ts equals the recomputed creation log for every asker (mismatches: 0)


  [OK ] profile.creation_performer equals the recomputed creation log for every asker (mismatches: 0)
  [OK ] profile.creation_logid equals the recomputed creation log for every asker (mismatches: 0)


  [OK ] profile.creation_source equals the recomputed creation log for every asker (mismatches: 0)
  [OK ] creation_action is one of create, create2, byemail, autocreate, forcecreatelocal or empty
  [OK ] every asker with no creation log was looked up in creation.jsonl and not found
'create' entries whose performer is the person's own name at fetch time or now: 99.94% (the rest are mostly renames: the log keeps the name used at registration)
seconds from registration to the creation log entry: {'count': 37013.0, 'mean': 8053.3, 'std': 1549184.5, 'min': 0.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 298043936.0}
  [OK ] for 99% of askers the creation log is within 2 s before to 10 s after registration (R7.7)


True

**What the output shows.** All 8 checks pass. The creation fields match the recomputed values for every asker. For
99.94% of `create` entries the person who did it has the person's own name; the rest are renamed accounts (the log
keeps the old name). The median gap from registration to the creation entry is 1 second. The mean (8,053 s) and the
maximum (about 9.4 years) look odd but come from one renamed old account whose entry is from 2015 (shown in section
6). The extra line `True` under the output is just the value returned by the last check and can be ignored.

### 3.5 Revert times (R7.8)

**What this checks.** For each edit tagged `mw-reverted`, M3 looked up the revision that undid it and its time. M7
uses this time so that a pre-question covariate counts only reverts that happened before the question, and an
outcome counts only reverts before the cutoff. The checks confirm each result belongs to a real reverted edit of the
same person and that the revert comes after the edit. Only edits within 30 days before the person's first question
or before the cutoff need a lookup ("candidates"): for older edits the revert cannot change which side of the
question or cutoff it falls on in practice.

The two raw files are combined, the full run winning over the sample run for the same edit (as in `revert_times.build`).
Each result must point to an edit of the same person that carries the `mw-reverted` tag, and a found revert must come
after the edit. An s6 edit got its tag when s6 was fetched (2026-04-23 to 24), so its revert must have happened
before 2026-04-25; a top-up edit was fetched in October, so its revert may fall after the cutoff (M7 ignores those).

In [9]:
rt = pd.concat([pd.DataFrame(read_jsonl(RAW3 / n)).assign(_file=n) for n in ("revert_times_sample.jsonl", "revert_times.jsonl")
                if (RAW3 / n).exists()], ignore_index=True).drop_duplicates("revid", keep="last")
rev_edits = edits[edits.tags.map(lambda x: "mw-reverted" in list(x))]
rj = rt.merge(edits[["revid", "uid", "timestamp", "source", "tags"]], on="revid", how="left", suffixes=("", "_edit"))
found = rj[rj.method.isin(["sha1", "undo_comment"])].copy()
found["lag_check"] = (pd.to_datetime(found.revert_ts, utc=True) - pd.to_datetime(found.ts, utc=True)).dt.total_seconds() / 3600
ck.add("each edit has at most one revert result after combining the files", rt.revid.is_unique)
ck.add("every revert result points to an edit in mentee_edits", rj.timestamp.notna().all())
ck.add("that edit carries the mw-reverted tag", rj.tags.dropna().map(lambda x: "mw-reverted" in list(x)).all())
ck.add("uid and edit time in the revert file equal the edit's", ((rj.uid == rj.uid_edit) & (rj.ts == rj.timestamp)).all())
ck.add("method is one of sha1, undo_comment, page_deleted, not_found, api_error",
       rt.method.isin(["sha1", "undo_comment", "page_deleted", "not_found", "api_error"]).all())
ck.add("every found revert has a revert revision and time", found.revert_revid.notna().all() and found.revert_ts.notna().all())
ck.add("the revert revision is later than the edit (larger revid)", (found.revert_revid > found.revid).all())
ck.add("the revert time is after the edit time", (found.revert_ts > found.ts).all())
ck.add("lag_hours equals revert time minus edit time (within 0.001 h)", (found.lag_check - found.lag_hours).abs().max() < 1e-3)
ck.add("s6 edits: the revert happened before s6 was fetched (before 2026-04-25)", (found[found.source == "s6"].revert_ts < "2026-04-25").all())
ck.add("lookups were made after the cutoff", (rt.fetched_at > CUT).all())

# coverage so far: candidates are mw-reverted edits within 30 days before the first question or before the cutoff
fq = pd.to_datetime(q.groupby("mentee_uid").q_ts.min(), utc=True)
ets = pd.to_datetime(rev_edits.timestamp, utc=True); w = pd.Timedelta(days=30)
near_q = (ets < rev_edits.uid.map(fq)) & (ets >= rev_edits.uid.map(fq) - w)
near_c = ets >= config.DATA_CUTOFF - w
cand = rev_edits[near_q | near_c]
cov_all = cand.revid.isin(rt.revid).mean()
print(f"\nmw-reverted edits {len(rev_edits):,} | candidates (30-day window) {len(cand):,} | with a result so far {int(cand.revid.isin(rt.revid).sum()):,} "
      f"({cov_all:.1%}) | results outside the candidate set (random sample of all mw-reverted edits) {int((~rt.revid.isin(cand.revid)).sum()):,}")
print("methods so far:", rt.method.value_counts().to_dict())
print("found reverts after the cutoff, by edit source:", found.assign(after=found.revert_ts >= CUT).groupby("source").after.sum().to_dict())

  [OK ] each edit has at most one revert result after combining the files
  [OK ] every revert result points to an edit in mentee_edits
  [OK ] that edit carries the mw-reverted tag
  [OK ] uid and edit time in the revert file equal the edit's
  [OK ] method is one of sha1, undo_comment, page_deleted, not_found, api_error
  [OK ] every found revert has a revert revision and time
  [OK ] the revert revision is later than the edit (larger revid)
  [OK ] the revert time is after the edit time
  [OK ] lag_hours equals revert time minus edit time (within 0.001 h)
  [OK ] s6 edits: the revert happened before s6 was fetched (before 2026-04-25)
  [OK ] lookups were made after the cutoff



mw-reverted edits 185,585 | candidates (30-day window) 36,981 | with a result so far 36,981 (100.0%) | results outside the candidate set (random sample of all mw-reverted edits) 1,233
methods so far: {'sha1': 36882, 'undo_comment': 771, 'page_deleted': 321, 'not_found': 240}
found reverts after the cutoff, by edit source: {'s6': 651, 'topup': 357}


**What the output shows.** All 11 checks pass. The lookup job is finished: all 36,981 candidate edits (of 185,585
reverted edits) have a result (100%), plus 1,233 extra edits from an earlier random sample used to choose the 30-day
window. Results: 36,882 found by sha1, 771 by an undo naming the edit, 321 on pages that were later deleted, and 240
not found. For 1,008 edits (651 s6, 357 top-up) the revert came after the cutoff; that is expected (s6 was
downloaded 3 days after the cutoff) and M7 does not count those reverts.

In [10]:
ck.report()


103 / 103 checks passed


**What the output shows.** All 103 invariants in sections 1 to 3 hold (`103 / 103 checks passed`).

## 4. Comparison with the previous report and with independent data

### 4.1 Previous report

`results/m3_report.json` from the last run of this notebook, next to the values now. Differences are expected only
after a pipeline rebuild.

In [11]:
now_vals = {"n_askers": len(prof), "source": prof.source.value_counts().to_dict(), "n_edits": len(edits),
            "creation_action": {str(k): int(v) for k, v in prof.creation_action.value_counts(dropna=False).items()}}
if prev_report:
    display(pd.DataFrame([{"key": k, "previous": json.dumps(prev_report.get(k)), "now": json.dumps(v), "same": prev_report.get(k) == v}
                          for k, v in now_vals.items()]))
else:
    print("no previous report")

,key,previous,now,same
0,n_askers,37095,37095,True
1,source,"{""s6"": 33642, ""s6+topup"": 2090, ""topup"": 1363}","{""s6"": 33642, ""s6+topup"": 2090, ""topup"": 1363}",True
2,n_edits,3522460,3522460,True
3,creation_action,"{""create"": 36335, ""byemail"": 367, ""autocreate"": 225, ""create2"": 85, ""None"": ...","{""create"": 36335, ""byemail"": 367, ""autocreate"": 225, ""create2"": 85, ""None"": ...",True


**What this checks and shows.** The main counts are compared with the saved report from the previous run. All four
are the same, so nothing changed between runs.

### 4.2 Independent edit data (val_mentee_contribs)

**What this checks.** A second, separately made download of the same people's edits is used as an outside check: if
our data were missing many edits, the activity covariates and outcomes would be too low.

`val_mentee_contribs.jsonl` was fetched separately by the 2SLS pipeline in June 2026, by user name, with the userid
on every edit. For people in both, we compare the revids before the cutoff. The validation data holds at most 5,000
edits per person and some people are incomplete, so many edits are "only in our data"; that is not an error. What
matters is the other direction: **are the validation edits in our data?** (coverage = shared / validation, target above
99%). Edits only in the validation data are mostly from renames (fetched under an old name) or pages deleted later.

In [12]:
if not config.EXTERNAL["val_mentee_contribs"].exists():
    print("skipped: the optional validation data (data/input/validation/) is not available")
    recall = both = tot = n_users = None
else:
    t = time.time()
    mine = edits.groupby("uid").revid.apply(set).to_dict()
    both = only_s6 = only_val = 0; n_users = 0
    per_user = []
    for line in open(config.EXTERNAL["val_mentee_contribs"], "rb"):
        r = orjson.loads(line)
        es = r.get("edits") or []
        if not es: continue
        uid = es[0].get("userid")
        if uid not in mine: continue
        val = {e["revid"] for e in es if e.get("timestamp", "") < CUT and e.get("userid") == uid}
        a = mine[uid]
        n_users += 1; both += len(a & val); only_s6 += len(a - val); only_val += len(val - a)
        per_user.append((uid, len(a), len(val), len(a & val)))
    tot = both + only_s6 + only_val
    recall = both / (both + only_val)
    print(f"people in both {n_users:,}; same revids {both:,}, only in our data {only_s6:,}, only in validation data {only_val:,}")
    print(f"share of validation edits found in our data: {recall:.4%} (target > 99%); agreement on the union: {both / tot:.4%}")
    pu = pd.DataFrame(per_user, columns=["uid", "n_mine", "n_val", "n_both"])
    pu["recall"] = pu.n_both / pu.n_val
    print("people with coverage < 99%:", int((pu.recall < .99).sum()), f"({time.time() - t:.0f} s)")
    display(pu.sort_values("recall").head(10))

people in both 30,173; same revids 1,516,010, only in our data 1,059,037, only in validation data 3,976
share of validation edits found in our data: 99.7384% (target > 99%); agreement on the union: 58.7823%
people with coverage < 99%: 23 (6 s)


,uid,n_mine,n_val,n_both,recall
28523,51803306,1,5,1,0.200000
26622,47209317,1,4,1,0.250000
1572,47907071,2,7,2,0.285714
11038,50957314,3,9,3,0.333333
2644,51901670,6,15,6,0.400000
15868,48833897,2105,3473,1703,0.490354
7322,50190228,1,2,1,0.500000
2989,48875733,1,2,1,0.500000
20974,49752795,13,18,13,0.722222
23401,52320595,3,4,3,0.750000


**What the output shows.** For the 30,173 people in both sources, 99.74% of the edits in the independent June
download are also in our data (target above 99%). Only 3,976 edits are missing from ours. The overall agreement on the
union (58.8%) is low only because the June download keeps at most 5,000 edits per person and is incomplete for some
people; that does not mean our data is wrong. 23 people are below 99% coverage. Most have only a handful of edits
(for example 1 of 5), which fits pages deleted later. One person (uid 48833897) lacks about 1,770 of 3,473 June
edits; this is the largest single gap and is worth a look by a reviewer, but it affects one asker out of 37,095.

### 4.3 Independent registration times (val_mentee_registration, June 2026, by uid)

**What this checks.** Registration time never changes, so it must agree with the profile for everyone found in both.
It is the basis of the account age covariate.

In [13]:
if not config.EXTERNAL["val_mentee_registration"].exists():
    print("skipped: the optional validation data (data/input/validation/) is not available")
    reg_agree = None
else:
    vreg = {r["userid"]: r.get("registration") for r in iter_jsonl(config.EXTERNAL["val_mentee_registration"]) if r.get("userid")}
    common = [u for u in P.index if u in vreg and vreg[u]]
    dis = [(u, P.at[u, "registration"], vreg[u]) for u in common if P.at[u, "registration"] != vreg[u]]
    print(f"askers in the June registration data: {len(common):,}; registration differs for {len(dis)}")
    print("first differences (uid, ours, June):", dis[:10])
    reg_agree = 1 - len(dis) / max(len(common), 1)

askers in the June registration data: 30,171; registration differs for 0
first differences (uid, ours, June): []


**What the output shows.** 30,171 askers are in the June data and the registration time agrees for all of them (0
differences).

## 5. Tracer samples and random samples: raw records next to processed values

### 5.1 Fixed tracer people

**What this checks.** These are the same fixed people every run, so a reader can open their Wikipedia pages and compare
by eye. Nothing is tested automatically here.

For each fixed tracer asker: the profile, the creation log, the first 5 edits, and links to open the contributions
page and the creation log on Wikipedia.

In [14]:
tr = tracer_frame()
for uid in [int(u) for u in tr[tr.kind == "mentee"].uid.dropna()]:
    r = P.loc[uid]
    print(f"\n### {r.current_name} (uid {uid})  source={r.source} late_fetch={r.late_fetch}  registered {r.registration}")
    print(f"  creation: {r.creation_action} at {r.creation_ts} by {r.creation_performer} (logid {r.creation_logid}, {r.creation_source})")
    print("  contributions:", contribs(r.current_name))
    if pd.notna(r.creation_logid): print("  creation log:", logid(r.creation_logid))
    e = edits[edits.uid == uid].sort_values("timestamp")
    print(f"  edits {len(e)} (articles, ns 0: {int((e.ns == 0).sum())}); logs {r.n_logs}; abuse {r.n_abuse}")
    display(e.head(5)[["timestamp", "ns", "revid", "sizediff", "tags"]])


### AlphaBetaGamma (uid 45433808)  source=s6+topup late_fetch=True  registered 2023-02-23T04:19:01Z
  creation: autocreate at 2023-02-23T04:19:02Z by AlphaBetaGamma (logid 143057393, log)
  contributions: https://en.wikipedia.org/wiki/Special:Contributions/AlphaBetaGamma?end=2026-04-20
  creation log: https://en.wikipedia.org/w/index.php?title=Special:Log&logid=143057393
  edits 19378 (articles, ns 0: 5828); logs 4292; abuse 70


,timestamp,ns,revid,sizediff,tags
93689,2023-02-28T03:33:11Z,4,1142025262,387,"[mobile edit, mobile web edit, advanced mobile edit, discussiontools, discus..."
93688,2023-02-28T03:42:37Z,3,1142029253,895,"[mobile edit, mobile web edit, advanced mobile edit, wikieditor]"
93687,2023-03-01T02:23:00Z,0,1142204836,8,"[mobile edit, mobile web edit, visualeditor, advanced mobile edit, newcomer ..."
93686,2023-03-01T02:42:22Z,0,1142207277,28,"[mobile edit, mobile web edit, visualeditor, advanced mobile edit, newcomer ..."
93685,2023-03-02T03:33:07Z,0,1142389225,0,"[mobile edit, mobile web edit, visualeditor, advanced mobile edit, newcomer ..."



### Couscous1812 (uid 50301938)  source=s6 late_fetch=False  registered 2025-09-21T23:26:13Z
  creation: create at 2025-09-21T23:26:14Z by Couscous1812 (logid 172797776, log)
  contributions: https://en.wikipedia.org/wiki/Special:Contributions/Couscous1812?end=2026-04-20
  creation log: https://en.wikipedia.org/w/index.php?title=Special:Log&logid=172797776
  edits 41 (articles, ns 0: 36); logs 4; abuse 3


,timestamp,ns,revid,sizediff,tags
393026,2025-09-21T23:28:25Z,0,1312665364,8,[wikieditor]
393025,2025-09-21T23:29:48Z,0,1312665524,-13,[wikieditor]
393024,2025-09-21T23:36:44Z,0,1312666370,7,"[mw-reverted, wikieditor]"
393023,2025-09-21T23:38:29Z,0,1312666576,14,[wikieditor]
393022,2025-09-26T14:14:36Z,3,1313486198,265,"[mentorship module question, discussiontools-added-comment]"



### Salicia7 (uid 49603274)  source=s6 late_fetch=False  registered 2025-04-29T19:19:43Z
  creation: create at 2025-04-29T19:19:44Z by Salicia7 (logid 169438347, log)
  contributions: https://en.wikipedia.org/wiki/Special:Contributions/Salicia7?end=2026-04-20
  creation log: https://en.wikipedia.org/w/index.php?title=Special:Log&logid=169438347
  edits 17 (articles, ns 0: 3); logs 4; abuse 0


...

,timestamp,ns,revid,sizediff,tags
1171809,2023-12-27T01:15:11Z,0,1191995611,-14,"[mobile edit, mobile web edit, visualeditor, newcomer task, newcomer task co..."
1171808,2023-12-27T01:21:11Z,0,1191996343,18,"[mobile edit, mobile web edit, visualeditor, newcomer task, newcomer task co..."
1171807,2023-12-27T01:27:14Z,0,1191997093,6,"[mobile edit, mobile web edit, visualeditor, newcomer task, newcomer task co..."
1171806,2023-12-27T04:34:56Z,0,1192017790,9,"[mobile edit, mobile web edit, visualeditor, newcomer task, newcomer task co..."
1171805,2023-12-27T04:46:23Z,2,1192018994,195,"[mobile edit, mobile web edit, visualeditor]"



### Saheed Aderemi Adewale Yussuf (uid 50375772)  source=s6 late_fetch=False  registered 2025-10-06T16:48:12Z
  creation: create at 2025-10-06T16:48:13Z by Saheed Aderemi Adewale Yussuf (logid 173122188, log)
  contributions: https://en.wikipedia.org/wiki/Special:Contributions/Saheed_Aderemi_Adewale_Yussuf?end=2026-04-20
  creation log: https://en.wikipedia.org/w/index.php?title=Special:Log&logid=173122188
  edits 26 (articles, ns 0: 0); logs 8; abuse 2


,timestamp,ns,revid,sizediff,tags
2505949,2025-10-06T17:46:21Z,3,1315438837,271,"[mobile edit, mobile web edit, mentorship module question, discussiontools-a..."
2505948,2025-10-06T17:47:09Z,3,1315438932,299,"[mobile edit, mobile web edit, mentorship module question, discussiontools-a..."
2505947,2025-10-06T18:10:00Z,3,1315442265,1602,"[mobile edit, mobile web edit, Possible self promotion in userspace, discuss..."
2505946,2025-10-06T18:12:43Z,3,1315442700,355,"[mobile edit, mobile web edit, discussiontools, discussiontools-source, mw-r..."
2505945,2025-10-06T18:14:39Z,3,1315442981,305,"[mobile edit, mobile web edit, discussiontools, discussiontools-source, mw-r..."



### 36Flames (uid 49563912)  source=s6+topup late_fetch=True  registered 2025-04-22T10:32:28Z
  creation: create at 2025-04-22T10:32:29Z by 36Flames (logid 169277654, log)
  contributions: https://en.wikipedia.org/wiki/Special:Contributions/36Flames?end=2026-04-20
  creation log: https://en.wikipedia.org/w/index.php?title=Special:Log&logid=169277654
  edits 176 (articles, ns 0: 48); logs 8; abuse 1


,timestamp,ns,revid,sizediff,tags
1526624,2025-05-12T06:26:58Z,4,1290006731,276,"[discussiontools-added-comment, wikieditor]"
1526623,2025-05-12T08:44:56Z,4,1290019459,118,"[discussiontools, discussiontools-reply, discussiontools-source, discussiont..."
1526622,2025-05-12T08:53:27Z,4,1290020058,380,[wikieditor]
1526621,2025-06-04T05:37:45Z,118,1293879721,1024,"[visualeditor-switched, wikieditor]"
1526620,2025-06-04T05:57:03Z,118,1293881326,682,"[visualeditor, editcheck-newreference]"


**What the output shows.** Each tracer has a creation entry within 2 seconds of registration, and their first
edits come after registration. Many first edits carry `mw-reverted` (newcomers' early edits are often undone), and the
question edit shows up with the tag `mentorship module question`. One tracer's account was `autocreate` (it was made
by logging in from another Wikimedia wiki); such people are excluded by the sample rule in M4. Very active tracers
(thousands of edits) are `s6+topup`, because their s6 history stopped early and was completed in October.

### Helper for the random examples

`person_check(uid)` reads the person's **raw** records (the raw s6 lines and/or the raw top-up record) and
recomputes, with its own simple code:

* the profile fields, next to the stored profile;
* the edits: raw records, those on or after the cutoff (must be dropped), duplicate revids (must be removed), and then
  count, first and last time, and the revid set, next to the merged table; then every field of every edit;
* the number of log entries and abuse filter hits;
* the account creation log record, shown raw, next to the `creation_*` fields.

In [15]:
FULL = ds.dataset(M3 / "mentee_edits").to_table(filter=ds.field("uid").isin(sorted(SAMPLE_UIDS))).to_pandas()
ECOLS = ["parentid", "ns", "title", "timestamp", "size", "sizediff", "tags", "minor", "new", "comment"]
def fingerprint(ids):
    ids = sorted(int(i) for i in ids)
    return f"{len(ids)} ids, sum {sum(ids)}"
random_results = []

def person_check(uid):
    p = P.loc[uid]; tp = topup.get(uid)
    print("\n" + "=" * 110)
    print(f"uid {uid}  {p.current_name}  source={p.source}  {contribs(p.current_name)}")
    # profile
    if tp is not None:
        u = tp["user"]; print(f"raw top-up user record ({tp['_file']}, fetched {tp['fetched_at'][:19]}):", json.dumps(u, ensure_ascii=False)[:250])
        hand_p = {"name_at_fetch": u.get("name"), "registration": u.get("registration"), "editcount": u.get("editcount"),
                  "blocked": "blockid" in u, "groups": list(u.get("groups") or []), "late_fetch": True}
    else:
        r = s6_prof_raw[s6_name[uid]]
        print("raw s6 profile line:", json.dumps({k: r.get(k) for k in ("username", "userid", "registration", "editcount", "blocked", "groups")}, ensure_ascii=False))
        hand_p = {"name_at_fetch": r["username"], "registration": r.get("registration"), "editcount": r.get("editcount"),
                  "blocked": bool(r.get("blocked")), "groups": list(r.get("groups") or []), "late_fetch": False}
    if not hand_p["registration"]: hand_p["registration"] = (reg_now.get(str(uid)) or {}).get("registration")
    hand_p["current_name"] = (reg_now.get(str(uid)) or {}).get("name")
    ok = compare({k: clean(v) for k, v in hand_p.items()}, {k: clean(p[k]) for k in hand_p}, "profile: raw vs processed")
    if uid in stale: print(f"stale.json: {stale[uid]}  (stale_edits.jsonl holds {len(stale_edits.get(uid, []))} edits)")

    # edits
    raw = [(e, "s6 raw") for e in s6_full.get(uid, [])] + [(e, tp["_file"]) for e in (tp or {}).get("contribs") or []]
    after = [e for e, _ in raw if e["timestamp"] >= CUT]
    uniq = {}
    for e, src in raw:
        if e["timestamp"] < CUT and e["revid"] not in uniq: uniq[e["revid"]] = (e, src)
    n_dup = sum(1 for e, _ in raw if e["timestamp"] < CUT) - len(uniq)
    m = FULL[FULL.uid == uid].set_index("revid")
    print(f"raw edit records {len(raw)} (s6 {len(s6_full.get(uid, []))}, top-up {len(raw) - len(s6_full.get(uid, []))}); "
          f"on or after the cutoff {len(after)}; duplicate revids {n_dup}")
    ts_raw = [e["timestamp"] for e, _ in uniq.values()]
    hand_e = {"n_edits": len(uniq), "rows in edit table": len(uniq), "first edit": min(ts_raw, default=None), "last edit": max(ts_raw, default=None),
              "revid set": fingerprint(uniq), "edits on or after cutoff": 0}
    data_e = {"n_edits": int(p.n_edits), "rows in edit table": len(m), "first edit": m.timestamp.min() if len(m) else None,
              "last edit": m.timestamp.max() if len(m) else None, "revid set": fingerprint(m.index), "edits on or after cutoff": int((m.timestamp >= CUT).sum())}
    ok &= compare(hand_e, data_e, "edits: raw vs merged")
    nbad = Counter()
    for rid, (e, src) in uniq.items():
        if rid not in m.index: continue
        row = m.loc[rid]
        rawv = {"parentid": e.get("parentid"), "ns": e.get("ns"), "title": e.get("title"), "timestamp": e.get("timestamp"), "size": e.get("size"),
                "sizediff": e.get("sizediff"), "tags": list(e.get("tags") or []), "minor": bool(e.get("minor")), "new": bool(e.get("new")), "comment": e.get("comment")}
        for f in ECOLS:
            if clean(rawv[f]) != clean(row[f]): nbad[f] += 1
        if row["source"] != ("s6" if src == "s6 raw" else "topup"): nbad["source"] += 1
    print("edit fields that differ from raw (field: number of edits):", dict(nbad) or "none")
    ok &= not nbad
    show = sorted(uniq.values(), key=lambda x: x[0]["timestamp"])
    show = show[:2] + show[-2:] if len(show) > 4 else show
    print(pd.DataFrame([{"revid": e["revid"], "raw file": src, "raw time": e["timestamp"], "merged time": m.at[e["revid"], "timestamp"],
                         "raw ns": e.get("ns"), "merged ns": m.at[e["revid"], "ns"], "raw sizediff": e.get("sizediff"),
                         "merged sizediff": m.at[e["revid"], "sizediff"], "merged source": m.at[e["revid"], "source"], "link": diff(e["revid"])}
                        for e, src in show]).to_string(index=False))

    # logs and abuse filter hits
    rl = [(e[1], e[4]) for e in s6_log_rows if e[0] == uid] + [(e.get("logid"), e.get("timestamp")) for e in (tp or {}).get("logs") or []]
    ra = [(e.get("id"), e.get("timestamp")) for e in s6_abuse_full.get(uid, [])] + [(e.get("id"), e.get("timestamp")) for e in (tp or {}).get("abuse") or []]
    hand_c = {"n_logs": len({i for i, ts in rl if ts < CUT}), "n_abuse": len({i for i, ts in ra if ts < CUT})}
    ok &= compare(hand_c, {k: int(p[k]) for k in hand_c}, f"logs and abuse: raw records (logs {len(rl)}, abuse {len(ra)}, with duplicates and post-cutoff) vs counts")
    ab_m = abuse[abuse.uid == uid].set_index("abuse_id")
    raw_ab = {e.get("id"): e for e in s6_abuse_full.get(uid, []) + list((tp or {}).get("abuse") or []) if e.get("timestamp") < CUT}
    n_ab_bad = sum(clean(ab_m.at[i, f]) != clean(str(e.get("filter_id") or "") if f == "filter_id" else e.get(f))
                   for i, e in raw_ab.items() if i in ab_m.index for f in ("filter_id", "filter", "action", "result", "timestamp", "ns", "title"))
    print("abuse filter fields that differ from raw:", n_ab_bad); ok &= n_ab_bad == 0

    # account creation log
    cands = [dict(logid=e[1], type=e[2], action=e[3], timestamp=e[4], title=e[6], user=e[7], params=e[8], file="s6 raw") for e in s6_log_rows if e[0] == uid]
    cands += [dict(logid=e.get("logid"), type=e.get("type"), action=e.get("action"), timestamp=e.get("timestamp"), title=e.get("title"),
                   user=e.get("user"), params=e.get("params") or {}, file=tp["_file"]) for e in (tp or {}).get("logs") or []]
    nu = sorted([c for c in cands if c["type"] == "newusers" and str(c["params"].get("userid")) == str(uid) and c["timestamp"] < CUT],
                key=lambda c: (c["timestamp"], c["logid"]))
    if nu:
        c = nu[0]; print("raw newusers log record:", json.dumps(c, ensure_ascii=False)[:300])
        hand_k = {"creation_action": c["action"], "creation_ts": c["timestamp"], "creation_performer": c["user"], "creation_logid": c["logid"], "creation_source": "log"}
    else:
        ev = next((r["event"] for r in creation_raw.get(uid, []) if r.get("event")), None)
        print("no own newusers entry in the raw logs; creation.jsonl:", json.dumps(creation_raw.get(uid), ensure_ascii=False)[:300])
        hand_k = ({"creation_action": ev.get("action"), "creation_ts": ev.get("timestamp"), "creation_performer": ev.get("user"),
                   "creation_logid": ev.get("logid"), "creation_source": "lookup"} if ev else
                  {"creation_action": None, "creation_ts": None, "creation_performer": None, "creation_logid": None, "creation_source": "none"})
    ok &= compare(hand_k, {k: clean(p[k]) for k in hand_k}, "account creation: raw log record vs profile")
    if hand_k["creation_logid"]: print("  open the log entry:", logid(hand_k["creation_logid"]), f"| registration {p.registration}")
    random_results.append((p.source, uid, ok))
    return ok

### 5.2 Random s6 askers

People whose data comes only from the April s6 files.

**What this checks.** For each random person, the raw downloaded record and the processed table are shown side by
side (profile, edits, log and abuse counts, creation entry), and each comparison prints "all fields match" or a
mismatch.

In [16]:
for uid in SAMPLE["s6"]: person_check(uid)


uid 49614674  Turkiyegazeteleri  source=s6  https://en.wikipedia.org/wiki/Special:Contributions/Turkiyegazeteleri?end=2026-04-20
raw s6 profile line: {"username": "Turkiyegazeteleri", "userid": 49614674, "registration": "2025-05-01T21:41:16Z", "editcount": 16, "blocked": false, "groups": ["*", "user", "autoconfirmed"]}
profile: raw vs processed
        field               recomputed                  in data  match
name_at_fetch        Turkiyegazeteleri        Turkiyegazeteleri   True
 registration     2025-05-01T21:41:16Z     2025-05-01T21:41:16Z   True
    editcount                       16                       16   True
      blocked                    False                    False   True
       groups [*, user, autoconfirmed] [*, user, autoconfirmed]   True
   late_fetch                    False                    False   True
 current_name        Turkiyegazeteleri        Turkiyegazeteleri   True
  -> all fields match
raw edit records 8 (s6 8, top-up 0); on or after the cutoff 0;

**What the output shows.** For all 5 random s6 people every field matches the raw record, no edit is on or after the
cutoff, and there are no duplicate edits.

### 5.3 Random s6+topup askers

People whose s6 edit history was incomplete (`stale.json`). Expected: merged edits = s6 raw edits + top-up edits,
each revision once, and all of them before the cutoff.

In [17]:
for uid in SAMPLE["s6+topup"]: person_check(uid)


uid 46649515  00101984hjw  source=s6+topup  https://en.wikipedia.org/wiki/Special:Contributions/00101984hjw?end=2026-04-20
raw top-up user record (topup_stale.jsonl, fetched 2026-10-24T19:51:04): {"userid": 46649515, "name": "00101984hjw", "editcount": 2478, "registration": "2023-10-03T02:40:27Z", "groups": ["extendedconfirmed", "*", "user", "autoconfirmed"]}
profile: raw vs processed
        field                                  recomputed                                     in data  match
name_at_fetch                                 00101984hjw                                 00101984hjw   True
 registration                        2023-10-03T02:40:27Z                        2023-10-03T02:40:27Z   True
    editcount                                        2478                                        2478   True
      blocked                                       False                                       False   True
       groups [extendedconfirmed, *, user, autoconfirmed] [extende

edit fields that differ from raw (field: number of edits): none
     revid          raw file             raw time          merged time  raw ns  merged ns  raw sizediff  merged sizediff merged source                                                 link
1178357389            s6 raw 2023-10-03T04:28:02Z 2023-10-03T04:28:02Z       0          0            80               80            s6 https://en.wikipedia.org/w/index.php?diff=1178357389
1178357467            s6 raw 2023-10-03T04:28:50Z 2023-10-03T04:28:50Z       0          0             2                2            s6 https://en.wikipedia.org/w/index.php?diff=1178357467
1348744282 topup_stale.jsonl 2026-04-14T02:51:53Z 2026-04-14T02:51:53Z       0          0            -1               -1         topup https://en.wikipedia.org/w/index.php?diff=1348744282
1349166959 topup_stale.jsonl 2026-04-16T02:47:57Z 2026-04-16T02:47:57Z       0          0            32               32         topup https://en.wikipedia.org/w/index.php?diff=1349166

edit fields that differ from raw (field: number of edits): none
     revid          raw file             raw time          merged time  raw ns  merged ns  raw sizediff  merged sizediff merged source                                                 link
1280247183            s6 raw 2025-03-13T13:07:07Z 2025-03-13T13:07:07Z       3          3           435              435            s6 https://en.wikipedia.org/w/index.php?diff=1280247183
1280258780            s6 raw 2025-03-13T14:39:03Z 2025-03-13T14:39:03Z       3          3           277              277            s6 https://en.wikipedia.org/w/index.php?diff=1280258780
1344585090 topup_stale.jsonl 2026-03-21T10:10:00Z 2026-03-21T10:10:00Z       0          0             6                6         topup https://en.wikipedia.org/w/index.php?diff=1344585090
1344694688 topup_stale.jsonl 2026-03-22T00:50:57Z 2026-03-22T00:50:57Z       0          0            54               54         topup https://en.wikipedia.org/w/index.php?diff=1344694

**What the output shows.** For all 5 random s6+topup people every field matches. For example, the first person has
2,106 s6 edits plus 68 top-up edits; 1 of the 2,174 raw edits is after the cutoff and is correctly dropped, giving
2,173. The raw log and abuse counts include duplicates (the same entry from both sources) and these are correctly
counted once.

### 5.4 Random topup askers

People that s6 does not have at all; everything comes from the October top-up fetched by uid.

In [18]:
for uid in SAMPLE["topup"]: person_check(uid)
print("\nrandom people: all fields match for", sum(ok for *_, ok in random_results), "of", len(random_results),
      "" if all(ok for *_, ok in random_results) else "-> MISMATCH, see above")


uid 51250241  "matal2"  source=topup  https://en.wikipedia.org/wiki/Special:Contributions/%22matal2%22?end=2026-04-20
raw top-up user record (topup.jsonl, fetched 2026-10-24T19:52:17): {"userid": 51250241, "name": "\"matal2\"", "editcount": 2, "registration": "2025-12-10T16:20:30Z", "groups": ["*", "user"]}
profile: raw vs processed
        field           recomputed              in data  match
name_at_fetch             "matal2"             "matal2"   True
 registration 2025-12-10T16:20:30Z 2025-12-10T16:20:30Z   True
    editcount                    2                    2   True
      blocked                False                False   True
       groups            [*, user]            [*, user]   True
   late_fetch                 True                 True   True
 current_name             "matal2"             "matal2"   True
  -> all fields match
raw edit records 2 (s6 0, top-up 2); on or after the cutoff 0; duplicate revids 0
edits: raw vs merged
                   field           

logs and abuse: raw records (logs 1, abuse 0, with duplicates and post-cutoff) vs counts
  field  recomputed  in data  match
 n_logs           1        1   True
n_abuse           0        0   True
  -> all fields match
abuse filter fields that differ from raw: 0
raw newusers log record: {"logid": 175604594, "type": "newusers", "action": "create", "timestamp": "2025-12-24T19:45:04Z", "title": "User:Bowlerconspiracy", "user": "Bowlerconspiracy", "params": {"userid": 51484953}, "file": "topup.jsonl"}
account creation: raw log record vs profile
             field           recomputed              in data  match
   creation_action               create               create   True
       creation_ts 2025-12-24T19:45:04Z 2025-12-24T19:45:04Z   True
creation_performer     Bowlerconspiracy     Bowlerconspiracy   True
    creation_logid            175604594            175604594   True
   creation_source                  log                  log   True
  -> all fields match
  open the log entry: h

**What the output shows.** For all 5 random topup people everything comes from the October fetch and every field
matches. One of them is blocked, and the blocked flag is read correctly. Overall: all fields match for 15 of 15 random
people.

### 5.5 Random reverted edits with a revert time

**What this checks.** Five random reverts found by the lookup, with links to the edit and to the revision that undid
it, and a recomputation of the hours between them.

Open the two `diff` links: the first shows the newcomer's edit, the second the revision that undid it. The revert
revision should restore the page (sha1 method) or be an undo whose summary names the edit (undo_comment).

In [19]:
rs = pick(found, N_RANDOM, SEED)
title = ds.dataset(M3 / "mentee_edits").to_table(filter=ds.field("revid").isin([int(x) for x in rs.revid]), columns=["revid", "title", "tags"]).to_pandas().set_index("revid")
rev_ok = []
for r in rs.itertuples():
    print(f"\nedit {r.revid} by uid {r.uid} on '{title.at[r.revid, 'title']}' at {r.ts} (source {r.source}); tags {list(title.at[r.revid, 'tags'])}")
    print(f"  edit:   {diff(r.revid)}")
    print(f"  revert: {diff(int(r.revert_revid))}  ({oldid(int(r.revert_revid))}) at {r.revert_ts}, method {r.method}")
    print(f"  revert after the edit: {r.revert_ts > r.ts} | revert before the cutoff: {r.revert_ts < CUT}"
          + ("" if r.revert_ts < CUT else " (allowed in this file; M7 does not count it)"))
    hand = {"edit time": edits.loc[edits.revid == r.revid, "timestamp"].iloc[0], "uid": int(edits.loc[edits.revid == r.revid, "uid"].iloc[0]),
            "lag_hours": round((pd.Timestamp(r.revert_ts) - pd.Timestamp(r.ts)).total_seconds() / 3600, 3), "revert after edit": True}
    rev_ok.append(compare(hand, {"edit time": r.ts, "uid": int(r.uid), "lag_hours": r.lag_hours, "revert after edit": r.revert_ts > r.ts}))


edit 1312894978 by uid 50306510 on 'Talk:Plymouth Brethren Christian Church' at 2025-09-23T06:00:04Z (source topup); tags ['mw-reverted', 'wikieditor']
  edit:   https://en.wikipedia.org/w/index.php?diff=1312894978
  revert: https://en.wikipedia.org/w/index.php?diff=1313256650  (https://en.wikipedia.org/w/index.php?oldid=1313256650) at 2025-09-25T04:49:13Z, method sha1
  revert after the edit: True | revert before the cutoff: True
            field           recomputed              in data  match
        edit time 2025-09-23T06:00:04Z 2025-09-23T06:00:04Z   True
              uid             50306510             50306510   True
        lag_hours               46.819               46.819   True
revert after edit                 True                 True   True
  -> all fields match

edit 1345179651 by uid 40689896 on 'User:Captainbeetroots/sandbox' at 2026-03-24T19:03:50Z (source topup); tags ['visualeditor', 'mw-reverted']
  edit:   https://en.wikipedia.org/w/index.php?diff=1345179651

            field           recomputed              in data  match
        edit time 2024-12-17T20:27:32Z 2024-12-17T20:27:32Z   True
              uid             48052289             48052289   True
        lag_hours                0.154                0.154   True
revert after edit                 True                 True   True
  -> all fields match

edit 1347901361 by uid 48153652 on 'Bini (group)' at 2026-04-09T14:16:20Z (source topup); tags ['mw-undo', 'twinkle', 'mw-reverted']
  edit:   https://en.wikipedia.org/w/index.php?diff=1347901361
  revert: https://en.wikipedia.org/w/index.php?diff=1347913188  (https://en.wikipedia.org/w/index.php?oldid=1347913188) at 2026-04-09T15:26:29Z, method sha1
  revert after the edit: True | revert before the cutoff: True
            field           recomputed              in data  match
        edit time 2026-04-09T14:16:20Z 2026-04-09T14:16:20Z   True
              uid             48153652             48153652   True
        lag_hours        

**What the output shows.** All 5 match: the revert is after the edit, before the cutoff, and the stored hours (from
0.15 to 105 hours) equal the recomputed ones. All 5 were found by the sha1 method.

## 6. Edge-case lists

**What this checks.** Lists of unusual cases that each step wrote down, so a reader can see them instead of them being
silently dropped.

Issue files are written by each step. Note: a step that skips work already done (for example `collect_topup`,
which resumes) writes an empty `summary.json`, so older issue files can be leftovers from an earlier run. The table
marks files older than their summary; their content is still shown when it matches the current data.

In [20]:
for name in ("m3_index", "m3_collect", "m3"):
    d = config.INTERIM / name / "_issues"
    summ = json.loads((d / "summary.json").read_text())
    print(f"\n[{name}] summary.json: {({k: v['n'] for k, v in summ.items()})}")
    for f in sorted(d.glob("*.jsonl")):
        n = sum(1 for _ in open(f))
        older = f.stat().st_mtime < (d / "summary.json").stat().st_mtime - 60
        print(f"  {f.name:35s} {n:6,} rows  {'(older than summary.json: from an earlier run)' if older else ''}")
nf = read_jsonl(config.INTERIM / "m3_collect" / "_issues" / "creation_log_not_found.jsonl")
print(f"\ncreation_log_not_found: {len(nf)} rows; askers with creation_source='none' now: {int((prof.creation_source == 'none').sum())}; "
      f"same set: {set(r['ids']['uid'] for r in nf) == set(prof[prof.creation_source == 'none'].mentee_uid)}")
print("registration years of askers without a creation log (logs started in 2005):",
      pd.to_datetime(prof[prof.creation_source == "none"].registration, utc=True).dt.year.value_counts().sort_index().to_dict())
wa = read_jsonl(config.INTERIM / "m3" / "_issues" / "asker_without_activity.jsonl")
print(f"asker_without_activity.jsonl: {len(wa)} rows, of which now without data: {sum(P.at[r['ids']['uid'], 'source'] == 'none' for r in wa if r['ids']['uid'] in P.index)}")


[m3_index] summary.json: {'s6_profile_without_uid': 596}
  s6_profile_without_uid.jsonl           596 rows  

[m3_collect] summary.json: {}
  creation_log_not_found.jsonl            82 rows  (older than summary.json: from an earlier run)

[m3] summary.json: {}
  asker_without_activity.jsonl         1,519 rows  (older than summary.json: from an earlier run)

creation_log_not_found: 82 rows; askers with creation_source='none' now: 82; same set: True
registration years of askers without a creation log (logs started in 2005): {2004.0: 1, 2005.0: 4, 2006.0: 17, 2007.0: 11, 2008.0: 11, 2009.0: 14, 2010.0: 7, 2011.0: 4, 2012.0: 10, 2013.0: 1}
asker_without_activity.jsonl: 1519 rows, of which now without data: 0


**What the output shows.** 596 s6 profile rows have no user ID (names that no longer exist); they are not used. Two
issue files are marked as from an earlier run, because the later run skipped finished work; their content still
matches the current data. The 82 askers with no creation entry are exactly the 82 with `creation_source = none`, and
all registered in 2004 to 2013, when Wikipedia's account logs were incomplete. The 1,519 askers once listed as
"without activity" now all have data (0 left).

In [21]:
print("creation log far from registration (outside -2 s to +10 s):")
far = prof[prof.creation_gap_s.notna() & ~prof.creation_gap_s.between(-2, 10)]
display(far[["mentee_uid", "current_name", "registration", "creation_ts", "creation_gap_s", "creation_action", "creation_source"]].sort_values("creation_gap_s").head(15))
print("\nprofile edit counter lower than the edits we hold (editcount counts deleted edits too, so it is usually higher):",
      int((prof.editcount < prof.n_edits).sum()), "askers; by source", prof[prof.editcount < prof.n_edits].source.value_counts().to_dict())
low = prof[prof.editcount < prof.n_edits]
display(low.assign(diff=low.n_edits - low.editcount).sort_values("diff", ascending=False).head(8)[["mentee_uid", "current_name", "source", "registration", "editcount", "n_edits", "diff"]])
print("share of these registered before 2010 (old edit counters are known to be incomplete):", round(float((low.registration < "2010").mean()), 3))
print("\nstale.json reasons:", Counter(v["reason"] for v in stale.values()))
print("people found incomplete because s6 stopped at exactly 500 edits:", [u for u, v in stale.items() if v["reason"] == "capped_500"])
print("\nrevert lookups without a time (method, examples with links):")
for meth in ("page_deleted", "not_found", "api_error"):
    x = rt[rt.method == meth]
    print(f"  {meth}: {len(x)}", [diff(i) for i in pick(x, 3, SEED).revid])

creation log far from registration (outside -2 s to +10 s):


,mentee_uid,current_name,registration,creation_ts,creation_gap_s,creation_action,creation_source
326,41392002,WaterStar1058,2021-03-09T20:53:41Z,2021-03-09T20:53:52Z,11.0,create,log
318,41287084,Pettymars0193,2021-02-23T15:48:09Z,2021-02-23T15:48:32Z,23.0,create,log
9,738646,Tiagao~enwiki,2005-12-30T22:17:13Z,2015-06-11T12:16:09Z,298043936.0,autocreate,log



profile edit counter lower than the edits we hold (editcount counts deleted edits too, so it is usually higher): 286 askers; by source {'s6': 240, 's6+topup': 39, 'topup': 7}


,mentee_uid,current_name,source,registration,editcount,n_edits,diff
15997,48546149,Divinations,s6,2024-10-06T00:04:37Z,4864,5284,420
17149,48803042,Earth605,s6+topup,2024-11-24T15:29:13Z,3832,3921,89
21441,49453156,Zaptain United,s6,2025-03-31T01:48:13Z,5376,5449,73
11978,47760663,Swarleystinson88,s6,2024-04-26T12:10:41Z,261,292,31
26765,50050020,Bob the Guilder,s6,2025-07-31T04:13:54Z,548,566,18
26997,50079837,ChickpeaAnxiety,s6+topup,2025-08-06T14:36:33Z,1202,1219,17
27872,50179610,LASJM77,topup,2025-08-27T16:37:23Z,251,264,13
28483,50252551,SpongeBobMusicFan123,s6,2025-09-11T16:01:52Z,2385,2398,13


share of these registered before 2010 (old edit counters are known to be incomplete): 0.003

stale.json reasons: Counter({'newer_edits': 2085, 'capped_500': 5})
people found incomplete because s6 stopped at exactly 500 edits: [351337, 48759986, 30395827, 49914376, 43351275]

revert lookups without a time (method, examples with links):
  page_deleted: 321 ['https://en.wikipedia.org/w/index.php?diff=1278257949', 'https://en.wikipedia.org/w/index.php?diff=1349754158', 'https://en.wikipedia.org/w/index.php?diff=1318023953']
  not_found: 240 ['https://en.wikipedia.org/w/index.php?diff=1283688625', 'https://en.wikipedia.org/w/index.php?diff=1247638284', 'https://en.wikipedia.org/w/index.php?diff=1218868224']
  api_error: 0 []


**What the output shows.** Only 3 creation entries fall outside the 2 seconds before to 10 seconds after
registration: two at 11 and 23 seconds (still the right entry), and one renamed old account (`~enwiki` suffix) whose
entry is from 2015, when accounts were merged across wikis. For 286 askers the profile edit counter is lower than the
edits we hold. A reviewer may find this surprising: only 0.3% of them registered before 2010, so the usual "old
counters are incomplete" reason does not explain it. Wikipedia's edit counter is known to drift and is not used as a
covariate (the covariates are counted from the edit table), so it does not affect the results. 2,085 people were
re-fetched because they had newer edits than s6 held and 5 because s6 stopped at exactly 500 edits. There are no API
errors among the revert lookups.

## 7. Distributions

**What this checks.** Simple counts, to see whether the data looks reasonable and whether the late-fetched people
look different from the rest.

Account creation action (D10 uses it); registration year by source; and s6 versus top-up people, to see whether the
top-up people differ in a systematic way. Then the time from an edit to its revert.

In [22]:
print("account creation action:\n" + prof.creation_action.value_counts(dropna=False).to_string())
print("\nwhere the creation log came from:", prof.creation_source.value_counts().to_dict())
prof["reg_year"] = pd.to_datetime(prof.registration, utc=True, errors="coerce").dt.year
display(pd.crosstab(prof.reg_year, prof.source))
cmp = prof.groupby("source").agg(n=("mentee_uid", "size"), edits_median=("n_edits", "median"),
        edits_mean=("n_edits", "mean"), abuse_any=("n_abuse", lambda s: (s > 0).mean()), logs_median=("n_logs", "median"))
display(cmp)
print("\nedits per year by source:")
display(pd.crosstab(edits.timestamp.str[:4], edits.source))
print("\nhours from edit to revert (found so far):", found.lag_hours.quantile([.5, .9, .95, .99]).round(2).to_dict(),
      "| share over 30 days:", round(float((found.lag_hours > 24 * 30).mean()), 4))

account creation action:
creation_action
create              36335
byemail               367
autocreate            225
create2                85
None                   82
forcecreatelocal        1

where the creation log came from: {'log': 37012, 'none': 82, 'lookup': 1}


source,s6,s6+topup,topup
reg_year,,,
2004.0,1,0,0
2005.0,3,2,0
2006.0,13,3,1
2007.0,7,4,0
2008.0,9,2,0
2009.0,9,4,1
2010.0,7,0,0
2011.0,3,1,0
2012.0,8,2,0


,n,edits_median,edits_mean,abuse_any,logs_median
source,,,,,
s6,33642,3.0,55.019321,0.367606,1.0
s6+topup,2090,72.0,765.415789,0.784211,8.0
topup,1363,2.0,52.663977,0.434336,2.0



edits per year by source:


source,s6,topup
timestamp,,
2005,150,0
2006,155,1
2007,131,3
2008,118,68
2009,71,39
2010,960,41
2011,1035,35
2012,1344,27
2013,930,19



hours from edit to revert (found so far): {0.5: 1.64, 0.9: 154.73, 0.95: 356.98, 0.99: 2330.35} | share over 30 days: 0.0271


**What the output shows.** 36,335 of 37,095 accounts (98%) registered themselves (`create`). Almost all askers
registered in 2021 or later, when the mentorship feature started. The s6+topup people are much more active (median
72 edits versus 3) because they were re-fetched precisely for having many recent edits; the topup people look like
the s6 people (median 2 edits). This is why M8 reruns the analysis without late-fetched people. The large top-up
edit count in 2026 (200,103) is the recent edits added for those active people. Half of all reverts happen within
1.6 hours of the edit and only 2.7% take longer than 30 days, which supports the 30-day lookup window.

## 8. Summary (written to results/m3_report.json, used by M9)

**What this checks.** The key numbers are saved to a report file and a one-line summary of all checks is printed.

In [23]:
report = {"n_askers": len(prof), "source": prof.source.value_counts().to_dict(), "n_edits": len(edits),
          "creation_action": {str(k): int(v) for k, v in prof.creation_action.value_counts(dropna=False).items()},
          "val_recall": None if recall is None else round(recall, 6),
          "val_union_agreement": None if recall is None else round(both / tot, 6), "val_users": n_users}
(config.RESULTS / "m3_report.json").write_text(json.dumps(report, indent=1, ensure_ascii=False))
print(json.dumps(report, indent=1, ensure_ascii=False))
n_fail = sum(not ok for _, ok in ck.items)
print(f"\ninvariants: {len(ck.items) - n_fail} / {len(ck.items)} OK | random people matching raw: "
      f"{sum(ok for *_, ok in random_results)} / {len(random_results)} | random reverts: {sum(rev_ok)} / {len(rev_ok)} | "
      f"registration agrees with June data: {'n/a' if reg_agree is None else format(reg_agree, '.4%')} | revert-time coverage of candidates so far: {cov_all:.1%}")
print(f"notebook run time {time.time() - T0:.0f} s")

{
 "n_askers": 37095,
 "source": {
  "s6": 33642,
  "s6+topup": 2090,
  "topup": 1363
 },
 "n_edits": 3522460,
 "creation_action": {
  "create": 36335,
  "byemail": 367,
  "autocreate": 225,
  "create2": 85,
  "None": 82,
  "forcecreatelocal": 1
 },
 "val_recall": 0.997384,
 "val_union_agreement": 0.587823,
 "val_users": 30173
}

invariants: 103 / 103 OK | random people matching raw: 15 / 15 | random reverts: 5 / 5 | registration agrees with June data: 100.0000% | revert-time coverage of candidates so far: 100.0%
notebook run time 89 s


**What the output shows.** 103 of 103 invariants hold, all 15 random people and all 5 random reverts match their raw
records, registration times agree 100% with the June data, 99.74% of independent edits are found, and every edit that
needs a revert time has one. M3 is ready for use by M4, M5 and M7.